# Quantitative Structure–Activity Relationships I
### PHS 312 - Drug Discovery, Design, and Development
**Lecture 29 · Wednesday, November 11, 2026 · 50 minutes**

Pre-class reading: Patrick 7e, Chapter 18, Sections 18.1–18.3

**This activity takes 30-35 minutes.** Work in pairs. Run each cell with the
play button on its left, answer the question that follows, and keep going.
You will not write any code.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Spinner101/phs312-cadd-notebooks/blob/main/notebooks/PHS312_L29_QSAR_I.ipynb)

> **New:** Parts 2–4 are **lecture notes**: key ideas, the substituent-constant table drawn on one template, worked examples and step-by-step calculators. Read them before class or use them afterwards for review. The timed pair activity begins at Part 5; Parts 6–7 (statistics practice and a drug-series reference table) follow it.

---
### Before you start - what this notebook does with your work

* **Nothing you type is sent anywhere.** This notebook makes no network
  calls, writes no files, and does not connect to Google Drive. Your answers
  live in the memory of this browser tab and disappear when you close it.
* **Your instructor cannot see this page.** At the end you will get a short
  report code. You choose to paste it into Canvas - that is the only thing
  that reaches your instructor.
* **Do not type anything personal.** No patient information, no identifiers,
  nothing about a real person. Everything here is simulated.
* Opening this notebook signs you in with your Hampton Google account, which
  is covered by the university's agreement with Google. If you would rather
  not use a Google account, tell your instructor and you will be given an
  offline copy.
---

In [ ]:
#@title Set-up  -  run this cell first { display-mode: "form" }
#@markdown Press the play button once. Nothing is downloaded and nothing
#@markdown leaves this browser tab. Takes a few seconds.
import sys, types, textwrap

_tk = types.ModuleType("toolkit"); _tk.__dict__["__name__"] = "toolkit"
exec(compile("\"\"\"PHS 312 activity toolkit - assessment engine and simulation helpers.\n\nEverything runs in the browser session. Nothing is transmitted anywhere:\nno network calls, no file writes, no analytics. Scores exist only in the\nstudent's own Colab session until they choose to report them in Canvas.\n\"\"\"\nimport textwrap, hashlib, json\nfrom datetime import datetime\n\nW = 86\ndef _wrap(s, indent=\"\"):\n    out = []\n    for para in str(s).split(\"\\n\"):\n        out.extend(textwrap.wrap(para, W, initial_indent=indent,\n                                 subsequent_indent=indent) or [\"\"])\n    return \"\\n\".join(out)\n\ndef _rule(ch=\"-\"):\n    return ch * W\n\n\nclass Item:\n    \"\"\"One multiple-choice item with per-distractor feedback.\"\"\"\n    def __init__(self, qid, stem, options, key, why, bloom=\"\", silo=\"\",\n                 section=\"concept\", topic=\"\"):\n        if key not in options:\n            raise ValueError(f\"{qid}: key {key!r} not among options {list(options)}\")\n        missing = [k for k in options if k != key and k not in why]\n        if missing:\n            raise ValueError(f\"{qid}: no feedback written for distractor(s) {missing}\")\n        self.qid, self.stem, self.options = qid, stem, options\n        self.key, self.why = key, why\n        self.bloom, self.silo, self.section = bloom, silo, section\n        self.topic = topic\n\n\nclass Gradebook:\n    \"\"\"Tracks responses and reports performance. Local to this session.\"\"\"\n\n    SECTIONS = (\"precheck\", \"concept\", \"post\")\n    LABELS = {\"precheck\": \"Prior-knowledge check\",\n              \"concept\": \"Concept checks during the activity\",\n              \"post\": \"End-of-activity assessment\"}\n\n    def __init__(self, lecture, title):\n        self.lecture, self.title = lecture, title\n        self.items, self.responses = {}, {}\n        self.started = datetime.now()\n\n    # ---------- authoring ----------\n    def add(self, item):\n        if item.qid in self.items:\n            raise ValueError(f\"duplicate qid {item.qid}\")\n        self.items[item.qid] = item\n\n    def add_many(self, items):\n        for it in items:\n            self.add(it)\n\n    # ---------- student interaction ----------\n    def ask(self, qid):\n        \"\"\"Print the stem and options for an item.\"\"\"\n        it = self.items[qid]\n        print(_rule(\"=\"))\n        print(_wrap(f\"{qid}.  {it.stem}\"))\n        print()\n        for letter, text in it.options.items():\n            print(_wrap(f\"{letter}) {text}\", indent=\"    \"))\n        print(_rule(\"=\"))\n\n    def answer(self, qid, choice):\n        \"\"\"Record a response and give feedback appropriate to the section.\"\"\"\n        if qid not in self.items:\n            print(f\"[!] Unknown question id {qid!r}. Check the notebook.\")\n            return\n        it = self.items[qid]\n        choice = (choice or \"\").strip().upper()\n        if choice in (\"\", \"-\", \"SELECT\", \"NONE\"):\n            print(\"Choose an option from the dropdown, then press the play button again.\")\n            return\n        if choice not in it.options:\n            print(f\"[!] {choice!r} is not one of {list(it.options)}.\")\n            return\n\n        first = qid not in self.responses\n        if first:\n            self.responses[qid] = choice      # only the first response scores\n        correct = choice == it.key\n\n        if it.section == \"precheck\":\n            # No feedback yet: revealing it now would teach the answer\n            # before the activity and destroy the pre/post comparison.\n            n = len([q for q in self.responses\n                     if self.items[q].section == \"precheck\"])\n            total = len([q for q in self.items\n                         if self.items[q].section == \"precheck\"])\n            print(f\"Recorded. ({n} of {total} answered)\")\n            if n == total:\n                print(\"\\nAll set - run the next cell to begin the activity.\")\n            return\n\n        print(_rule())\n        if correct:\n            print(f\"Correct - {choice}.\")\n            print(_wrap(it.why.get(\"_correct\", \"\"), \"\"))\n        else:\n            print(f\"Not quite. You chose {choice}.\")\n            print(_wrap(it.why[choice]))\n            print()\n            print(_wrap(f\"The defensible answer is {it.key}. \"\n                        + it.why.get(\"_correct\", \"\")))\n        if not first:\n            kept = self.responses[qid]\n            print(f\"\\n(Your first response, {kept}, is the one on record - \"\n                  f\"this retry is for your own understanding.)\")\n        print(_rule())\n\n    # ---------- reporting ----------\n    def _score(self, section):\n        qs = [q for q, it in self.items.items() if it.section == section]\n        got = sum(1 for q in qs if self.responses.get(q) == self.items[q].key)\n        return got, len(qs)\n\n    def summary(self, show_pre_post=True):\n        print(_rule(\"=\"))\n        print(f\"  {self.lecture} - {self.title}\")\n        print(f\"  Performance summary\")\n        print(_rule(\"=\"))\n        any_missing = False\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            if tot == 0:\n                continue\n            answered = len([q for q, it in self.items.items()\n                            if it.section == sec and q in self.responses])\n            if answered < tot:\n                any_missing = True\n            pct = 100 * got / tot if tot else 0\n            bar = \"#\" * int(round(pct / 5)) + \".\" * (20 - int(round(pct / 5)))\n            print(f\"  {self.LABELS[sec]:<38} {got:>2}/{tot:<3} {pct:5.1f}%  [{bar}]\")\n            if answered < tot:\n                print(f\"      ({tot - answered} unanswered)\")\n        print(_rule())\n\n        if show_pre_post:\n            pre_g, pre_t = self._score(\"precheck\")\n            post_g, post_t = self._score(\"post\")\n            if pre_t and post_t:\n                pre = 100 * pre_g / pre_t\n                post = 100 * post_g / post_t\n                print(f\"  Before the activity: {pre:5.1f}%     \"\n                      f\"After the activity: {post:5.1f}%\")\n                if post >= pre:\n                    if pre < 100:\n                        g = (post - pre) / (100 - pre)\n                        print(f\"  Normalised learning gain  g = {g:.2f}\"\n                              f\"   {self._gain_note(g)}\")\n                    else:\n                        print(\"  You came in at ceiling on the prior check.\")\n                else:\n                    print(\"  Your post score is below your pre score - look back at the\")\n                    print(\"  concept checks you missed before you leave.\")\n                print(_rule())\n\n        self._missed()\n        if any_missing:\n            print(\"\\n  Some questions are unanswered. Scroll back and complete them\")\n            print(\"  before copying your report code.\")\n        return self\n\n    @staticmethod\n    def _gain_note(g):\n        if g >= 0.7: return \"(high)\"\n        if g >= 0.3: return \"(medium)\"\n        return \"(low - review before the exam)\"\n\n    def _missed(self):\n        missed = [(q, it) for q, it in self.items.items()\n                  if it.section != \"precheck\"\n                  and q in self.responses and self.responses[q] != it.key]\n        if not missed:\n            print(\"  Nothing missed in the graded sections.\")\n            return\n        print(\"  Worth reviewing before the exam:\")\n        for q, it in missed:\n            tag = f\"[{it.bloom}\" + (f\" | SILO {it.silo}\" if it.silo else \"\") + \"]\"\n            print(_wrap(f\"- {q} {tag}: {it.stem}\", indent=\"    \"))\n\n    def report_code(self, student_name=\"\"):\n        \"\"\"A short code the student pastes into Canvas.\n\n        This is a participation signal, not a secure exam score: it is\n        generated in the browser and a determined student could edit it.\n        Weight it accordingly.\n        \"\"\"\n        tags = {\"precheck\": \"PRE\", \"concept\": \"CON\", \"post\": \"POST\"}\n        parts = []\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            parts.append(f\"{tags[sec]}{got}/{tot}\" if tot else f\"{tags[sec]}-\")\n        body = f\"{self.lecture}|\" + \",\".join(parts)\n        digest = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper()\n        code = f\"{body}|{digest}\"\n        print(_rule(\"=\"))\n        print(\"  COPY THIS LINE INTO THE CANVAS ASSIGNMENT\")\n        print(_rule(\"=\"))\n        print(f\"  {code}\")\n        print(_rule(\"=\"))\n        print(_wrap(\"Nothing was sent anywhere. This code exists only in your \"\n                    \"browser until you paste it into Canvas yourself.\"))\n        return code\n\n\ndef verify_code(code):\n    \"\"\"Instructor-side check that a pasted code has not been casually edited.\"\"\"\n    try:\n        body, digest = code.strip().rsplit(\"|\", 1)\n        ok = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper() == digest.upper()\n        return ok, body\n    except Exception:\n        return False, \"\"\n\n\n    # ---------- diagnostic review (used by the L28 review session) ----------\ndef _topic_table(gb, guide=None, section=None):\n    \"\"\"Score by topic and say what to revisit. Attached to Gradebook below.\n\n    `section` limits the table to one section, so the mid-session diagnostic\n    does not count questions the student has not reached yet.\n    \"\"\"\n    rows = {}\n    for qid, it in gb.items.items():\n        if not it.topic:\n            continue\n        if section and it.section != section:\n            continue\n        got = gb.responses.get(qid)\n        r = rows.setdefault(it.topic, [0, 0, 0])\n        r[1] += 1\n        if got is not None:\n            r[2] += 1\n            if got == it.key:\n                r[0] += 1\n    if not rows:\n        print(\"No topic-tagged questions have been answered yet.\")\n        return {}\n    print(_rule(\"=\"))\n    print(\"  Where you stand, by lecture\")\n    print(_rule(\"=\"))\n    print(f\"  {'lecture':<9}{'answered':>10}{'correct':>9}{'':>4}\")\n    weak = []\n    for t in sorted(rows):\n        ok, total, ans = rows[t]\n        pct = (100.0 * ok / ans) if ans else 0.0\n        bar = \"#\" * int(round(pct / 10)) + \".\" * (10 - int(round(pct / 10)))\n        flag = \"\"\n        if ans == 0:\n            bar, flag = \"-\" * 10, \"  not attempted\"\n        elif pct < 67:\n            flag = \"  <- revisit\"\n            weak.append(t)\n        print(f\"  {t:<9}{ans:>5} of {total:<4}{ok:>5}   {bar}{flag}\")\n    print()\n    if weak:\n        print(\"  Before the exam, go back to:\")\n        for t in weak:\n            where = (guide or {}).get(t, \"\")\n            print(f\"    {t}  {where}\" if where else f\"    {t}\")\n    else:\n        done = sum(r[2] for r in rows.values())\n        total = sum(r[1] for r in rows.values())\n        if done < total:\n            print(f\"  Nothing below two thirds so far, with {total - done} \"\n                  f\"question(s) still unanswered.\")\n        else:\n            print(\"  No lecture is below two thirds. Review your missed \"\n                  \"questions above,\\n  then re-run the activities you found \"\n                  \"slowest rather than re-reading.\")\n    print(_rule(\"=\"))\n    return {t: dict(correct=v[0], answered=v[2], total=v[1]) for t, v in rows.items()}\n\n\nGradebook.topic_report = _topic_table\n", "toolkit.py", "exec"), _tk.__dict__)
sys.modules["toolkit"] = _tk
Item, Gradebook, verify_code = _tk.Item, _tk.Gradebook, _tk.verify_code

_sb = types.ModuleType("swissbio"); _sb.__dict__["__name__"] = "swissbio"
exec(compile("# -*- coding: utf-8 -*-\n\"\"\"Curated bioisosteric replacement data for PHS 312 Lecture 31.\n\nSOURCE\n    SwissBioisostere, https://www.swissbioisostere.ch\n    Molecular Modelling Group, SIB Swiss Institute of Bioinformatics and\n    the University of Lausanne. Underlying data derived from ChEMBL 36.\n\nLICENCE\n    CC-BY 4.0  https://creativecommons.org/licenses/by/4.0/\n    Redistribution and adaptation permitted with attribution.\n\nCHANGES MADE (required by CC-BY)\n    Retrieved 20 October 2026 through the SwissBioisostere web interface.\n    Two single-attachment-point (\"side chain\") fragment queries were run:\n    carboxylic acid [*:1]C(=O)O  and primary amide [*:1]C(=O)N.\n    The full result tables (7,714 and 6,955 replacements) were reduced to the\n    highest-frequency entries shown below. Column names were shortened and\n    values were not otherwise altered. No structures were recomputed.\n\nCITE\n    Cuozzo A., Daina A., Perez M.A.S., Michielin O., Zoete V.\n    SwissBioisostere: molecular replacements for drug discovery guided by\n    updated structural, bioactivity and physico-chemical data accessible\n    through a reshaped Web interface. Nucleic Acids Research, 2021.\n    Wirth M., Zoete V., Michielin O., Sauer W.H.B. SwissBioisostere: a\n    database of molecular replacements for ligand design. Nucleic Acids\n    Research 2013, 41(D1), D1137-D1143. doi:10.1093/nar/gks1059\n\nCOLUMNS\n    smiles  replacement fragment, [*:1] marks the attachment point\n    freq    number of matched molecular pairs in which this replacement occurs\n    rank    rank by frequency within that query\n    better / similar / worse\n            how the replacement changed measured activity across those pairs\n    dlogp, dtpsa, dmw\n            mean change in calculated logP, topological PSA and molecular\n            weight on making the replacement\n\"\"\"\n\nFIELDS = (\"smiles\", \"freq\", \"rank\", \"better\", \"similar\", \"worse\",\n          \"dlogp\", \"dtpsa\", \"dmw\")\n\n# common names for the fragments students meet in Patrick 7e 13.3.7\nNAMES = {\n    \"COC(=O)[*:1]\": \"methyl ester\",\n    \"NC(=O)[*:1]\": \"primary amide\",\n    \"CCOC(=O)[*:1]\": \"ethyl ester\",\n    \"O[*:1]\": \"hydroxyl\",\n    \"C[*:1]\": \"methyl\",\n    \"OC[*:1]\": \"hydroxymethyl\",\n    \"O=C(O)C[*:1]\": \"acetic acid (homologated)\",\n    \"N#C[*:1]\": \"nitrile\",\n    \"CNC(=O)[*:1]\": \"N-methyl amide\",\n    \"CO[*:1]\": \"methoxy\",\n    \"N[*:1]\": \"amine\",\n    \"CN(C)C(=O)[*:1]\": \"N,N-dimethyl amide\",\n    \"O=C(NO)[*:1]\": \"hydroxamic acid\",\n    \"c1ccc([*:1])cc1\": \"phenyl\",\n    \"F[*:1]\": \"fluorine\",\n    \"Cl[*:1]\": \"chlorine\",\n    \"n1n[nH]c([*:1])n1\": \"tetrazol-5-yl\",\n    \"FC(F)(F)[*:1]\": \"trifluoromethyl\",\n    \"n1nc([*:1])n[nH]1\": \"tetrazol-5-yl (tautomer)\",\n    \"O=[N+]([O-])[*:1]\": \"nitro\",\n    \"NS(=O)(=O)[*:1]\": \"sulfonamide\",\n    \"CS(=O)(=O)NC(=O)[*:1]\": \"acylsulfonamide\",\n    \"O=C(O)CC[*:1]\": \"propanoic acid (homologated)\",\n    \"O=C(NCCO)[*:1]\": \"N-(2-hydroxyethyl) amide\",\n    \"CC(=O)N[*:1]\": \"acetamido\",\n    \"Br[*:1]\": \"bromine\",\n    \"CS(=O)(=O)[*:1]\": \"methylsulfonyl\",\n    \"O=C(N1CCOCC1)[*:1]\": \"morpholine amide\",\n    \"CC(=O)[*:1]\": \"acetyl\",\n    \"CN(C)[*:1]\": \"dimethylamino\",\n    \"O=C(O)[*:1]\": \"carboxylic acid\",\n    \"CCNC(=O)[*:1]\": \"N-ethyl amide\",\n    \"O=C(NC1CC1)[*:1]\": \"N-cyclopropyl amide\",\n    \"CS(=O)(=O)N[*:1]\": \"methanesulfonamido\",\n}\n\n# ---- carboxylic acid: what replaces [*:1]C(=O)O  (7,714 replacements total)\nACID_TOTAL = 7714\nACID = [\n (\"COC(=O)[*:1]\", 4530, 1, 1238, 2353, 939, 0.09, -11.00, 14.03),\n (\"NC(=O)[*:1]\", 2781, 2, 1000, 1334, 447, -0.60, 5.79, -0.99),\n (\"CCOC(=O)[*:1]\", 2449, 3, 703, 1024, 722, 0.48, -11.00, 28.06),\n (\"O[*:1]\", 1626, 4, 604, 793, 229, 0.01, -17.07, -28.01),\n (\"C[*:1]\", 1478, 5, 620, 616, 242, 0.61, -37.30, -29.98),\n (\"OC[*:1]\", 1144, 6, 395, 546, 203, -0.20, -17.07, -13.98),\n (\"O=C(O)C[*:1]\", 956, 7, 253, 530, 173, -0.08, 0.00, 14.02),\n (\"N#C[*:1]\", 877, 8, 346, 353, 178, 0.17, -13.51, -19.00),\n (\"CNC(=O)[*:1]\", 818, 9, 298, 412, 108, -0.34, -8.20, 13.04),\n (\"CO[*:1]\", 788, 10, 331, 334, 123, 0.31, -28.07, -13.98),\n (\"N[*:1]\", 721, 11, 201, 368, 152, -0.51, -11.28, -29.00),\n (\"CN(C)C(=O)[*:1]\", 692, 12, 230, 331, 131, 0.00, -16.99, 27.07),\n (\"O=C(NO)[*:1]\", 654, 13, 440, 157, 57, -0.58, 12.03, 15.02),\n (\"c1ccc([*:1])cc1\", 635, 14, 271, 262, 102, 1.96, -37.30, 32.09),\n (\"F[*:1]\", 625, 15, 265, 259, 101, 0.44, -37.30, -26.02),\n (\"Cl[*:1]\", 604, 16, 276, 240, 88, 0.95, -37.30, -9.57),\n (\"n1n[nH]c([*:1])n1\", 545, 17, 105, 325, 115, -0.52, 17.16, 24.03),\n (\"FC(F)(F)[*:1]\", 503, 18, 214, 206, 83, 1.32, -37.30, 23.98),\n (\"n1nc([*:1])n[nH]1\", 484, 19, 135, 256, 93, -0.51, 17.16, 24.03),\n (\"O=[N+]([O-])[*:1]\", 384, 20, 166, 167, 51, 0.21, 5.84, 0.99),\n (\"NS(=O)(=O)[*:1]\", 382, 21, 157, 168, 57, -1.05, 22.86, 35.07),\n (\"CS(=O)(=O)NC(=O)[*:1]\", 369, 22, 36, 226, 107, -1.01, 25.94, 77.11),\n (\"O=C(O)CC[*:1]\", 348, 23, 112, 191, 45, 0.32, 0.00, 28.06),\n (\"O=C(NCCO)[*:1]\", 341, 24, 148, 142, 51, -0.98, 12.03, 43.07),\n (\"CC(=O)N[*:1]\", 331, 25, 111, 160, 60, 0.26, -8.20, 13.04),\n (\"Br[*:1]\", 330, 26, 153, 125, 52, 1.06, -37.30, 34.89),\n (\"CS(=O)(=O)[*:1]\", 327, 27, 132, 141, 54, -0.43, -3.16, 34.08),\n (\"O=C(N1CCOCC1)[*:1]\", 315, 28, 93, 151, 71, -0.23, -7.76, 69.10),\n (\"CC(=O)[*:1]\", 296, 29, 120, 118, 58, 0.50, -20.23, -1.97),\n (\"CN(C)[*:1]\", 283, 30, 93, 116, 74, 0.08, -34.06, -0.94),\n]\n\n# ---- primary amide: what replaces [*:1]C(=O)N  (6,955 replacements total)\nAMIDE_TOTAL = 6955\nAMIDE = [\n (\"O=C(O)[*:1]\", 2781, 1, 447, 1334, 1000, 0.60, -5.79, 0.99),\n (\"N#C[*:1]\", 2384, 2, 586, 1189, 609, 1.04, -19.30, -18.01),\n (\"CNC(=O)[*:1]\", 2202, 3, 192, 1591, 419, 0.26, -13.99, 14.03),\n (\"O[*:1]\", 1622, 4, 398, 1003, 221, 0.11, -22.86, -27.02),\n (\"COC(=O)[*:1]\", 1269, 5, 259, 699, 311, 0.68, -16.79, 15.02),\n (\"C[*:1]\", 1238, 6, 361, 668, 209, 1.21, -43.09, -28.99),\n (\"CN(C)C(=O)[*:1]\", 1112, 7, 128, 590, 394, 0.61, -22.78, 28.06),\n (\"OC[*:1]\", 1060, 8, 233, 651, 176, 0.51, -22.86, -13.00),\n (\"CO[*:1]\", 1003, 9, 235, 554, 214, 0.91, -33.86, -12.99),\n (\"F[*:1]\", 720, 10, 195, 360, 165, 1.04, -43.09, -25.04),\n (\"Cl[*:1]\", 669, 11, 262, 281, 126, 1.55, -43.09, -8.58),\n (\"N[*:1]\", 646, 12, 151, 332, 163, 0.08, -17.07, -28.01),\n (\"FC(F)(F)[*:1]\", 635, 13, 194, 272, 169, 1.92, -43.09, 24.97),\n (\"CCOC(=O)[*:1]\", 612, 14, 146, 344, 122, 1.08, -16.79, 29.03),\n (\"CS(=O)(=O)[*:1]\", 599, 15, 90, 403, 106, 0.17, -8.95, 35.06),\n (\"CC(=O)N[*:1]\", 546, 16, 80, 349, 117, 0.26, -13.99, 14.03),\n (\"O=C(NCCO)[*:1]\", 532, 17, 42, 385, 105, -0.38, 6.24, 44.05),\n (\"c1ccc([*:1])cc1\", 476, 18, 128, 246, 102, 2.37, -43.09, 33.07),\n (\"O=C(N1CCOCC1)[*:1]\", 443, 19, 45, 250, 148, 0.37, -13.55, 70.09),\n (\"CC(=O)[*:1]\", 441, 20, 112, 235, 94, 1.10, -26.02, -0.98),\n (\"CCNC(=O)[*:1]\", 438, 21, 65, 269, 104, 0.65, -13.99, 28.06),\n (\"O=C(NC1CC1)[*:1]\", 431, 22, 94, 246, 91, 0.80, -13.99, 40.06),\n (\"NS(=O)(=O)[*:1]\", 382, 23, 72, 232, 78, -0.45, 17.07, 36.06),\n (\"CS(=O)(=O)N[*:1]\", 359, 24, 58, 225, 76, 0.27, 3.08, 50.08),\n (\"Br[*:1]\", 339, 25, 137, 146, 56, 1.66, -43.09, 35.88),\n]\n\nQUERIES = {\n    \"carboxylic acid\": dict(query=\"[*:1]C(=O)O\", total=ACID_TOTAL, rows=ACID),\n    \"primary amide\":   dict(query=\"[*:1]C(=O)N\", total=AMIDE_TOTAL, rows=AMIDE),\n}\n\n\ndef table(which=\"carboxylic acid\"):\n    q = QUERIES[which]\n    return [dict(zip(FIELDS, r), name=NAMES.get(r[0], r[0])) for r in q[\"rows\"]]\n\n\ndef success_rate(row):\n    \"\"\"Fraction of matched pairs where activity did not get worse.\"\"\"\n    n = row[\"better\"] + row[\"similar\"] + row[\"worse\"]\n    return (row[\"better\"] + row[\"similar\"]) / n if n else 0.0\n", "swissbio.py", "exec"), _sb.__dict__)
sys.modules["swissbio"] = _sb
swissbio = _sb


_sm = types.ModuleType("sims"); _sm.__dict__["__name__"] = "sims"
exec(compile("\"\"\"PHS 312 simulations. NumPy + matplotlib only - no install, no network.\n\nEvery function returns the numbers it plotted so the notebook can quiz on them.\n\"\"\"\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\n\nPRIMARY, SECOND, MUTED = \"#0B6E8F\", \"#C2410C\", \"#6B7280\"\n\n\ndef _tw(text, width=76, indent=\"\"):\n    import textwrap\n    return \"\\n\".join(textwrap.fill(p, width, initial_indent=indent,\n                                   subsequent_indent=indent)\n                     for p in text.split(\"\\n\"))\n\n\ndef _fig(w=9.5, h=4.4):\n    f, ax = plt.subplots(figsize=(w, h))\n    ax.grid(alpha=.25); ax.set_axisbelow(True)\n    for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    return f, ax\n\n\n# ---------------------------------------------------------------- L22\ndef force_field_terms(bond_k=300., bond_r0=1.52, ang_k=60., ang_t0=109.5,\n                      tor_V=14., vdw_eps=0.3, vdw_sigma=3.6, q1q2=-0.3):\n    \"\"\"Each force-field term plotted over its own coordinate.\"\"\"\n    f, axes = plt.subplots(1, 4, figsize=(15, 3.4))\n    r = np.linspace(1.1, 2.1, 200); Es = bond_k * (r - bond_r0) ** 2\n    t = np.linspace(80, 140, 200);  Eb = ang_k * np.radians(t - ang_t0) ** 2\n    d = np.linspace(-180, 180, 361); Et = tor_V / 2 * (1 + np.cos(np.radians(3 * d)))\n    x = np.linspace(3.0, 8.0, 300)\n    Ev = 4 * vdw_eps * ((vdw_sigma / x) ** 12 - (vdw_sigma / x) ** 6)\n    Ec = 332. * q1q2 / x\n    for ax, (xx, yy, xl, ti) in zip(axes, [\n            (r, Es, \"bond length (A)\", \"stretch\"),\n            (t, Eb, \"bond angle (deg)\", \"bend\"),\n            (d, Et, \"torsion (deg)\", \"torsion\"),\n            (x, Ev + Ec, \"separation (A)\", \"vdW + electrostatic\")]):\n        ax.plot(xx, yy, color=PRIMARY, lw=2)\n        ax.set_title(ti, fontsize=10); ax.set_xlabel(xl, fontsize=8)\n        ax.grid(alpha=.25); ax.axhline(0, color=MUTED, lw=.8)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    axes[0].set_ylabel(\"energy (kcal/mol)\", fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(stretch_min=float(r[Es.argmin()]), torsion_barrier=float(Et.max()),\n                vdw_min_sep=float(x[(Ev + Ec).argmin()]),\n                vdw_min_E=float((Ev + Ec).min()))\n\n\n# ---------------------------------------------------------------- L23\ndef torsion_profile(V3=14.0, gauche_strain=3.8, step=10, start_angle=100.):\n    \"\"\"Butane-like relaxed torsion profile, plus where minimisation stops.\"\"\"\n    d = np.arange(-180, 181, step)\n    E = V3 / 2 * (1 + np.cos(np.radians(3 * d)))\n    E += gauche_strain * np.exp(-((d - 60) / 32.) ** 2)\n    E += gauche_strain * np.exp(-((d + 60) / 32.) ** 2)\n    E += 2.2 * gauche_strain * np.exp(-(d / 30.) ** 2)\n    E = E - E.min()\n    # steepest descent from start_angle on this surface\n    fine = np.arange(-180, 180.01, 0.5)\n    Ef = np.interp(fine, d, E)\n    i = int(np.argmin(np.abs(fine - start_angle)))\n    while True:\n        lo, hi = max(i - 1, 0), min(i + 1, len(fine) - 1)\n        j = lo if Ef[lo] < Ef[i] else (hi if Ef[hi] < Ef[i] else i)\n        if j == i: break\n        i = j\n    f, ax = _fig()\n    ax.plot(d, E, \"-o\", ms=3.5, color=PRIMARY, label=\"relaxed scan\")\n    ax.axvline(start_angle, color=MUTED, ls=\":\", label=f\"start {start_angle:.0f}\")\n    ax.plot([fine[i]], [Ef[i]], \"v\", ms=12, color=SECOND,\n            label=f\"minimisation stops at {fine[i]:.0f} deg\")\n    ax.set_xlabel(\"C1-C2-C3-C4 dihedral (degrees)\")\n    ax.set_ylabel(\"relative steric energy (kJ/mol)\")\n    ax.set_xticks(range(-180, 181, 60)); ax.legend(fontsize=8)\n    plt.tight_layout(); plt.show()\n    mins = [float(d[k]) for k in range(1, len(d) - 1)\n            if E[k] < E[k - 1] and E[k] < E[k + 1]]\n    return dict(global_min=float(d[E.argmin()]), local_minima=mins,\n                barrier_at_0=float(E[d == 0][0]), stops_at=float(fine[i]))\n\n\ndef search_cost(n_rot=7, step=30):\n    n = int(round(360 / step)) ** n_rot\n    secs = n * 0.05\n    return dict(conformations=n, hours=secs / 3600,\n                years=secs / 3600 / 24 / 365)\n\n\n# ---------------------------------------------------------------- L24\ndef anneal(T=300., n_steps=4000, seed=0):\n    \"\"\"Metropolis walk on a double-well surface: does it escape the local min?\n\n    Global minima sit at +/-180 deg (0 kJ/mol); a local minimum sits at\n    +60 deg (4 kJ/mol) behind a ~26 kJ/mol barrier. At low T the walker is\n    trapped; raising T lets it cross.\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    PLATEAU, DEEP, SHALLOW, W = 30.0, 30.0, 26.0, 26.0\n\n    def E(a):\n        a = np.asarray(a, float)\n        return (PLATEAU\n                - DEEP * np.exp(-((a - 180) / W) ** 2)\n                - DEEP * np.exp(-((a + 180) / W) ** 2)\n                - SHALLOW * np.exp(-((a - 60) / W) ** 2))\n\n    pos = 60.0\n    visited = [pos]\n    kT = max(T, 1.) * 0.0083145                      # kJ/mol per K\n    for _ in range(n_steps):\n        trial = ((pos + rng.normal(0, 20) + 180) % 360) - 180\n        dE = float(E(trial) - E(pos))\n        if dE < 0 or rng.random() < np.exp(-min(dE / kT, 700)):\n            pos = trial\n        visited.append(pos)\n    visited = np.array(visited)\n\n    second_half = visited[len(visited) // 2:]\n    near_global = float(np.mean(np.abs(np.abs(second_half) - 180) < 50) * 100)\n    ever = bool(np.any(np.abs(np.abs(visited) - 180) < 50))\n\n    x = np.linspace(-180, 180, 721)\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))\n    a1.plot(x, E(x), color=PRIMARY, lw=2)\n    a1.set_xlabel(\"dihedral (deg)\"); a1.set_ylabel(\"energy (kJ/mol)\")\n    a1.set_title(\"energy surface\")\n    a1.plot([60], [E(60)], \"o\", color=SECOND, ms=10)\n    a1.annotate(\"start (local, 4 kJ/mol)\", (60, E(60) + 1.5), fontsize=8, ha=\"center\")\n    a1.annotate(\"global (0 kJ/mol)\", (180, E(180) + 1.5), fontsize=8, ha=\"right\")\n    a2.hist(visited, bins=72, color=PRIMARY, alpha=.85)\n    a2.set_xlabel(\"dihedral visited (deg)\"); a2.set_ylabel(\"count\")\n    a2.set_title(f\"T = {T:.0f} K   -   \"\n                 + (\"escaped the local minimum\" if near_global > 15\n                    else \"stayed trapped\"))\n    for ax in (a1, a2):\n        ax.grid(alpha=.25)\n        for sp in (\"top\", \"right\"): ax.spines[sp].set_visible(False)\n    plt.tight_layout(); plt.show()\n    return dict(escaped=near_global > 15, pct_near_global=round(near_global, 1),\n                ever_reached_global=ever, final=round(float(visited[-1]), 1))\n\n\ndef rmsd_overlay(dx=0.0, dy=0.0, rot_deg=0.0):\n    \"\"\"Rigid-body move of a probe onto a template; reports RMSD.\"\"\"\n    tmpl = np.array([[0., 0.], [1.4, 0.3], [2.6, -0.4], [3.9, 0.2], [1.2, 1.7]])\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    probe = (tmpl - tmpl.mean(0)) @ R.T + tmpl.mean(0) + np.array([dx, dy])\n    rmsd = float(np.sqrt(((probe - tmpl) ** 2).sum(1).mean()))\n    f, ax = _fig(6.4, 4.6)\n    ax.plot(*tmpl.T, \"o-\", color=PRIMARY, ms=11, lw=2, label=\"template (rigid active)\")\n    ax.plot(*probe.T, \"s--\", color=SECOND, ms=10, lw=2, label=\"your analogue\")\n    for p, q in zip(tmpl, probe):\n        ax.plot([p[0], q[0]], [p[1], q[1]], color=MUTED, lw=.9, ls=\":\")\n    ax.set_title(f\"RMSD = {rmsd:.2f} A\"); ax.legend(fontsize=8); ax.set_aspect(\"equal\")\n    plt.tight_layout(); plt.show()\n    return dict(rmsd=rmsd, good=rmsd < 0.5)\n\n\n# ---------------------------------------------------------------- L25\n# A snug pocket, 12.0 x 8.0 A, holding a ligand that spans 5.0 A, with a\n# gap in the wall where a polar residue sits. Orientation matters: turned\n# across the pocket the ligand cannot fit without clashing.\n_A, _B = 6.0, 4.1\n_TH = np.linspace(0, 2 * np.pi, 25)[:-1]\n_TH = np.array([t for t in _TH\n                if abs(np.degrees(t) - 90) > 16])          # gap for the residue\nSITE = np.c_[_A * np.cos(_TH), _B * np.sin(_TH)]           # cavity wall atoms\nHBOND_POINT = np.array([0.0, 3.7])                         # polar partner\nLIGAND = np.array([[-2.5, -0.3], [-0.8, 0.3], [0.9, 0.2],\n                   [2.5, -0.4], [0.0, 1.3]])               # last atom is polar\nCLASH_AT, CONTACT_AT, HBOND_AT = 3.0, 3.9, 2.7\n\n\ndef dock_pose(x=0.0, y=0.0, rot_deg=0.0, show=True):\n    \"\"\"Place a rigid 5-atom ligand in a 2-D pocket it very nearly fills.\n\n    Score = -(shape contacts + H-bond) + steric clash penalty, so a MORE\n    NEGATIVE score is a better pose. A pose outside the pocket earns nothing\n    and tends to zero; a pose driven into the wall goes sharply positive.\n    \"\"\"\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    pose = LIGAND @ R.T + np.array([x, y])\n\n    d = np.linalg.norm(pose[:, None, :] - SITE[None, :, :], axis=2)\n    nearest = d.min(1)\n    bad = nearest < CLASH_AT\n    clash = float(np.sum(np.clip(CLASH_AT - nearest, 0, None) ** 2) * 12)\n    contact = float(np.sum(np.exp(-((nearest - CONTACT_AT) / 1.25) ** 2)) * 2.6)\n    hb_dist = float(np.linalg.norm(pose[4] - HBOND_POINT))\n    hbond = float(6.0 * np.exp(-((hb_dist - HBOND_AT) / 0.95) ** 2))\n    score = -(contact + hbond) + clash\n\n    if show:\n        f, ax = _fig(7.0, 5.4)\n        _full = np.linspace(0, 2 * np.pi, 160)\n        ax.fill(_A * np.cos(_full), _B * np.sin(_full), color=\"#DCE9EF\",\n                zorder=0)\n        ax.plot(np.append(SITE[:, 0], SITE[0, 0]),\n                np.append(SITE[:, 1], SITE[0, 1]), \"o\", color=PRIMARY,\n                ms=15, label=\"binding-site wall\", zorder=1)\n        ax.plot(*HBOND_POINT, \"*\", ms=22, color=\"#B45309\",\n                label=\"polar residue\", zorder=3)\n        chain = pose[:4]\n        ax.plot(chain[:, 0], chain[:, 1], \"-\", color=SECOND, lw=2.6,\n                alpha=.8, zorder=4)\n        ax.plot([chain[1, 0], pose[4, 0]], [chain[1, 1], pose[4, 1]], \"-\",\n                color=SECOND, lw=2.6, alpha=.8, zorder=4)\n        ok = ~bad\n        if ok.any():\n            ax.plot(pose[ok][:, 0], pose[ok][:, 1], \"o\", ms=14,\n                    color=SECOND, label=\"ligand atom\", zorder=5)\n        if bad.any():\n            ax.plot(pose[bad][:, 0], pose[bad][:, 1], \"X\", ms=19,\n                    color=\"#DC2626\", label=\"steric clash\", zorder=6)\n        ax.set_aspect(\"equal\")\n        ax.set_xlim(-9.5, 9.5); ax.set_ylim(-7, 7)\n        ax.legend(fontsize=8, loc=\"lower left\")\n        ax.set_title(f\"score {score:+.1f}    \"\n                     f\"contacts {contact:.1f}   H-bond {hbond:.1f}   \"\n                     f\"clash {clash:.1f}\")\n        plt.tight_layout(); plt.show()\n    return dict(score=round(score, 2), contact=round(contact, 2),\n                hbond=round(hbond, 2), clash=round(clash, 2),\n                clashing_atoms=int(bad.sum()))\n\n\n# ---------------------------------------------------------------- L26\ndef score_vs_truth(cutoff=-7.0, seed=3):\n    \"\"\"Docking scores against crystallographic truth: the ranking problem.\"\"\"\n    rng = np.random.default_rng(seed)\n    n = 60\n    active = rng.random(n) < 0.25\n    rmsd = np.where(active, rng.gamma(2, .9, n), rng.gamma(5, 1.1, n))\n    score = -4.5 - 3.2 * active - rng.normal(0, 1.5, n)\n    picked = score <= cutoff\n    tp = int(np.sum(picked & active)); fp = int(np.sum(picked & ~active))\n    tot_act = int(active.sum())\n    hit = 100 * tp / max(picked.sum(), 1)\n    base = 100 * tot_act / n\n    f, ax = _fig(8.6, 4.6)\n    ax.scatter(score[~active], rmsd[~active], s=46, color=MUTED,\n               label=\"decoy\", alpha=.75)\n    ax.scatter(score[active], rmsd[active], s=56, color=PRIMARY,\n               label=\"true active\", alpha=.9)\n    ax.axvline(cutoff, color=SECOND, lw=2, ls=\"--\",\n               label=f\"your cutoff {cutoff:+.1f}\")\n    ax.axhline(2.0, color=\"#B45309\", lw=1.2, ls=\":\",\n               label=\"2 A - pose counts as correct\")\n    ax.set_xlabel(\"docking score (more negative = better)\")\n    ax.set_ylabel(\"RMSD to crystal pose (A)\")\n    ax.legend(fontsize=8)\n    ax.set_title(f\"picked {int(picked.sum())} compounds: {tp} actives, {fp} decoys\"\n                 f\"   |   hit rate {hit:.0f}% vs {base:.0f}% at random\")\n    plt.tight_layout(); plt.show()\n    return dict(picked=int(picked.sum()), true_actives_found=tp, decoys=fp,\n                total_actives=tot_act, hit_rate=round(hit, 1),\n                random_rate=round(base, 1),\n                enrichment=round(hit / base, 2) if base else 0.0)\n\n\n# ---------------------------------------------------------------- L27\ndef de_novo(h_bond_group=\"amidine\", near_group=\"phenyl\", far_group=\"isopropyl\",\n            linker_bonds=5, fill_site=False):\n    \"\"\"Grow and link fragments into a site with one polar and two greasy pockets.\"\"\"\n    HB = {\"amidine\": 5.0, \"tetrazole\": 4.2, \"phenol\": 3.0, \"methyl\": 0.2,\n          \"isopropyl\": 0.0}\n    GREASE = {\"phenyl\": 3.6, \"cyclohexyl\": 3.9, \"isopropyl\": 2.6,\n              \"methyl\": 1.2, \"amidine\": -1.5, \"tetrazole\": -1.8, \"phenol\": 0.6}\n    SYNTH = {\"amidine\": 1.0, \"tetrazole\": 1.6, \"phenol\": 0.3, \"phenyl\": 0.2,\n             \"cyclohexyl\": 0.5, \"isopropyl\": 0.1, \"methyl\": 0.0}\n    inter = HB.get(h_bond_group, 0) + GREASE.get(near_group, 0) + GREASE.get(far_group, 0)\n    ideal = 6\n    strain = 0.9 * abs(linker_bonds - ideal)\n    synth = SYNTH.get(h_bond_group, 0) + SYNTH.get(near_group, 0) + SYNTH.get(far_group, 0) \\\n        + 0.25 * max(linker_bonds - 8, 0)\n    overfill = 3.0 if fill_site else 0.0\n    total = inter - strain - synth - overfill\n    labels = [\"H-bond region\", \"near pocket\", \"far pocket\",\n              \"linker strain\", \"synthetic cost\", \"over-fill\"]\n    vals = [HB.get(h_bond_group, 0), GREASE.get(near_group, 0),\n            GREASE.get(far_group, 0), -strain, -synth, -overfill]\n    f, ax = _fig(9.2, 3.8)\n    cols = [PRIMARY if v >= 0 else SECOND for v in vals]\n    ax.barh(labels, vals, color=cols)\n    ax.axvline(0, color=\"k\", lw=.9)\n    ax.set_xlabel(\"contribution to design score\")\n    ax.set_title(f\"design score {total:+.1f}\"\n                 + (\"   (over-filled: no room left to optimise)\" if fill_site else \"\"))\n    plt.tight_layout(); plt.show()\n    return dict(score=round(total, 2), interactions=round(inter, 2),\n                strain=round(strain, 2), synthetic_cost=round(synth, 2),\n                overfill_penalty=overfill)\n\n\n# ---------------------------------------------------------------- L29 / L30\nCOUMARIN = dict(\n    X=[\"H\", \"Me\", \"OH\", \"OMe\", \"OCF3\", \"NH2\", \"F\", \"Cl\", \"CF3\", \"CN\", \"NO2\"],\n    pIC50=np.array([8.36, 8.38, 8.01, 8.44, 7.94, 7.46, 8.55, 8.48, 8.24, 7.97, 8.59]),\n    pi=np.array([0., .49, -.67, -.09, 1.02, -1.23, .14, .71, .88, -.57, -.26]),\n    Vdw=np.array([.08, 1.01, .49, 1.49, 1.60, .67, .36, 1.07, 1.11, 1.14, 1.20]),\n    F=np.array([0., -.04, .28, .25, .37, .02, .42, .40, .37, .50, .66]))\n\n_PI = {\"H\": 0., \"Me\": .56, \"Cl\": .71, \"NO2\": -.28, \"OMe\": -.02, \"NH2\": -1.23, \"F\": .14}\n_MR = {\"H\": .103, \"Me\": .565, \"Cl\": .603, \"NO2\": .736, \"OMe\": .787, \"NH2\": .542, \"F\": .092}\n_F = {\"H\": 0., \"Me\": .01, \"Cl\": .42, \"NO2\": .65, \"OMe\": .29, \"NH2\": .08, \"F\": .45}\n_R = {\"H\": 0., \"Me\": -.18, \"Cl\": -.19, \"NO2\": .13, \"OMe\": -.56, \"NH2\": -.74, \"F\": -.39}\n_TAC = [(\"tacrine\", \"H\", \"H\", 6.59), (\"I\", \"Me\", \"H\", 5.09), (\"II\", \"H\", \"Me\", 7.00),\n        (\"III\", \"Cl\", \"H\", 6.26), (\"IV\", \"H\", \"Cl\", 8.00), (\"V\", \"NO2\", \"H\", 5.52),\n        (\"VI\", \"H\", \"NO2\", 7.55), (\"VII\", \"H\", \"OMe\", 6.45), (\"VIII\", \"NH2\", \"H\", 5.42),\n        (\"IX\", \"H\", \"F\", 7.06), (\"X\", \"Cl\", \"Cl\", 6.32), (\"XI\", \"OMe\", \"OMe\", 5.29)]\n\ndef tacrine_table():\n    rows = []\n    for cid, X, Y, p in _TAC:\n        rows.append(dict(cpd=cid, X=X, Y=Y, pIC50=p, piX=_PI[X], piY=_PI[Y],\n                         MRX=_MR[X], MRY=_MR[Y], F_XY=_F[X] + _F[Y], R_XY=_R[X] + _R[Y]))\n    return rows\n\n\ndef _lstsq(X, y):\n    X = np.column_stack([np.asarray(X, float).reshape(len(y), -1), np.ones(len(y))])\n    beta, *_ = np.linalg.lstsq(X, y, rcond=None)\n    pred = X @ beta\n    ss_res = float(((y - pred) ** 2).sum()); ss_tot = float(((y - y.mean()) ** 2).sum())\n    r2 = 1 - ss_res / ss_tot\n    n, p = len(y), X.shape[1]\n    s = float(np.sqrt(ss_res / (n - p))) if n > p else float(\"nan\")\n    return beta, pred, r2, s\n\n\ndef qsar_single(param=\"pi\", quadratic=False):\n    \"\"\"One descriptor at a time on the coumarin MAO-B series.\"\"\"\n    y = COUMARIN[\"pIC50\"]; x = COUMARIN[param]\n    cols = np.column_stack([x, x ** 2]) if quadratic else x\n    beta, pred, r2, s = _lstsq(cols, y)\n    f, ax = _fig(7.6, 4.4)\n    ax.scatter(x, y, s=70, color=PRIMARY, zorder=3)\n    for xi, yi, t in zip(x, y, COUMARIN[\"X\"]):\n        ax.annotate(t, (xi, yi), fontsize=8, xytext=(4, 4), textcoords=\"offset points\")\n    xs = np.linspace(x.min(), x.max(), 200)\n    ys = beta[0] * xs + beta[1] * xs ** 2 + beta[2] if quadratic else beta[0] * xs + beta[1]\n    ax.plot(xs, ys, color=SECOND, lw=2,\n            label=(\"quadratic\" if quadratic else \"linear\") + f\"   r = {np.sqrt(max(r2,0)):.3f}   s = {s:.3f}\")\n    ax.set_xlabel({\"pi\": \"substituent hydrophobicity  pi\",\n                   \"Vdw\": \"van der Waals volume\", \"F\": \"inductive factor  F\"}[param])\n    ax.set_ylabel(\"pIC50 (MAO-B)\"); ax.legend(fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(param=param, quadratic=quadratic,\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3))\n\n\ndef qsar_multi(use=(\"MRX\",), show=True):\n    \"\"\"Add descriptors and watch r rise while leave-one-out gets worse.\"\"\"\n    rows = tacrine_table()\n    y = np.array([r[\"pIC50\"] for r in rows])\n    X = np.column_stack([[r[c] for r in rows] for c in use])\n    beta, pred, r2, s = _lstsq(X, y)\n    loo = []\n    for i in range(len(y)):\n        m = np.ones(len(y), bool); m[i] = False\n        b, _, _, _ = _lstsq(X[m], y[m])\n        loo.append(float(np.dot(np.append(X[i], 1.0), b)))\n    loo = np.array(loo)\n    q2 = 1 - float(((y - loo) ** 2).sum()) / float(((y - y.mean()) ** 2).sum())\n    if show:\n        f, ax = _fig(5.6, 5.2)\n        ax.scatter(pred, y, s=72, color=PRIMARY, label=\"fitted\", zorder=3)\n        ax.scatter(loo, y, s=52, color=SECOND, marker=\"s\",\n                   label=\"left-out prediction\", alpha=.85, zorder=3)\n        lim = [min(y.min(), pred.min(), loo.min()) - .4,\n               max(y.max(), pred.max(), loo.max()) + .4]\n        ax.plot(lim, lim, \"k--\", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)\n        ax.set_xlabel(\"predicted pIC50\"); ax.set_ylabel(\"observed pIC50\")\n        ax.set_title(\" + \".join(use) + f\"\\nr = {np.sqrt(max(r2,0)):.3f}    \"\n                     f\"s = {s:.3f}    q2 = {q2:.3f}\")\n        ax.legend(fontsize=8); plt.tight_layout(); plt.show()\n    return dict(descriptors=list(use), n=len(y), k=len(use),\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3),\n                q2=round(q2, 3), compounds_per_parameter=round(len(y) / len(use), 1))\n\n\n# ---------------------------------------------------------------- L31\nBIOISOSTERE = {\n    \"carboxylic acid (COOH)\": dict(pKa=4.2, logP=0.0, MW=45, HBD=1, HBA=2, note=\"parent\"),\n    \"tetrazole\":              dict(pKa=4.9, logP=0.5, MW=69, HBD=1, HBA=4, note=\"classic acid mimic\"),\n    \"acylsulfonamide\":        dict(pKa=4.5, logP=0.6, MW=121, HBD=1, HBA=4, note=\"acid mimic, bulkier\"),\n    \"2,6-difluorophenol\":     dict(pKa=7.1, logP=1.9, MW=130, HBD=1, HBA=1, note=\"lipophilic acid mimic\"),\n    \"hydroxamic acid\":        dict(pKa=9.0, logP=-0.4, MW=61, HBD=2, HBA=3, note=\"weaker acid, metal binder\"),\n    \"primary amide (CONH2)\":  dict(pKa=None, logP=-1.5, MW=44, HBD=2, HBA=2, note=\"neutral\"),\n    \"methyl ester\":           dict(pKa=None, logP=0.6, MW=59, HBD=0, HBA=2, note=\"neutral, hydrolysable\"),\n}\n\ndef bioisostere_swap(replacement=\"tetrazole\", need_acidic=True, need_permeable=True):\n    p = BIOISOSTERE[\"carboxylic acid (COOH)\"]; r = BIOISOSTERE[replacement]\n    keeps_acid = r[\"pKa\"] is not None and r[\"pKa\"] < 6.5\n    more_lipo = r[\"logP\"] > p[\"logP\"]\n    verdict = []\n    if need_acidic:\n        verdict.append((\"keeps the ionised acid at pH 7.4\", keeps_acid))\n    if need_permeable:\n        verdict.append((\"raises lipophilicity for permeability\", more_lipo))\n    verdict.append((\"molecular weight increase under 60 Da\", r[\"MW\"] - p[\"MW\"] < 60))\n    f, ax = _fig(8.8, 3.6)\n    labels = [\"pKa\", \"logP\", \"MW (Da)\", \"HBD\", \"HBA\"]\n    par = [p[\"pKa\"] if p[\"pKa\"] else 0, p[\"logP\"], p[\"MW\"] / 20, p[\"HBD\"], p[\"HBA\"]]\n    rep = [r[\"pKa\"] if r[\"pKa\"] else 0, r[\"logP\"], r[\"MW\"] / 20, r[\"HBD\"], r[\"HBA\"]]\n    i = np.arange(len(labels)); w = .38\n    ax.bar(i - w / 2, par, w, label=\"carboxylic acid (parent)\", color=MUTED)\n    ax.bar(i + w / 2, rep, w, label=replacement, color=PRIMARY)\n    ax.set_xticks(i); ax.set_xticklabels(labels)\n    ax.set_title(f\"{replacement} - {r['note']}   (MW shown /20)\")\n    ax.legend(fontsize=8); ax.axhline(0, color=\"k\", lw=.8)\n    plt.tight_layout(); plt.show()\n    for text, ok in verdict:\n        print((\"  MET      \" if ok else \"  NOT MET  \") + text)\n    return dict(replacement=replacement, pKa=r[\"pKa\"], logP=r[\"logP\"],\n                dMW=r[\"MW\"] - p[\"MW\"], requirements_met=sum(1 for _, o in verdict if o),\n                requirements_total=len(verdict))\n\n\n# ---------------------------------------------------------------- L32\ndef prodrug_kinetics(promoiety=\"ethyl ester\", esterase_activity=1.0, hours=8.0):\n    \"\"\"First-order activation: prodrug -> active drug, and patient variability.\"\"\"\n    K = {\"ethyl ester\": 0.45, \"5-indanyl ester (EWG)\": 1.60,\n         \"pivaloyl ester (hindered)\": 0.10, \"amide\": 0.02, \"no promoiety\": 0.0}\n    k = K[promoiety] * max(esterase_activity, 0.0)\n    t = np.linspace(0, hours, 400)\n    pro = np.exp(-k * t); act = 1 - pro\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3))\n    a1.plot(t, pro * 100, color=MUTED, lw=2, label=\"prodrug (inactive)\")\n    a1.plot(t, act * 100, color=PRIMARY, lw=2.4, label=\"active drug\")\n    a1.set_xlabel(\"hours\"); a1.set_ylabel(\"% of dose\"); a1.legend(fontsize=8)\n    a1.set_title(f\"{promoiety}   k = {k:.2f} /h\")\n    for lab, mult, col in [(\"low enzyme (x0.3)\", .3, SECOND),\n                           (\"typical (x1)\", 1., PRIMARY),\n                           (\"high enzyme (x3)\", 3., \"#15803D\")]:\n        a2.plot(t, (1 - np.exp(-K[promoiety] * mult * t)) * 100, lw=2.2,\n                color=col, label=lab)\n    a2.set_xlabel(\"hours\"); a2.set_ylabel(\"% converted to active drug\")\n    a2.set_title(\"same prodrug, three patients\"); a2.legend(fontsize=8)\n    for ax in (a1, a2):\n        ax.grid(alpha=.25); ax.set_ylim(-3, 103)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    plt.tight_layout(); plt.show()\n    half = float(np.log(2) / k) if k > 0 else float(\"inf\")\n    at2 = float((1 - np.exp(-k * 2)) * 100)\n    spread = float(abs((1 - np.exp(-K[promoiety] * 3 * 2)) -\n                       (1 - np.exp(-K[promoiety] * .3 * 2))) * 100)\n    return dict(promoiety=promoiety, k_per_h=round(k, 3),\n                half_life_h=(round(half, 2) if np.isfinite(half) else None),\n                pct_active_at_2h=round(at2, 1),\n                patient_spread_at_2h_pct=round(spread, 1))\n\n\n# ---------------------------------------------------------------- L22 extra\ndef method_choice(atoms=20, want=\"partial charges and an electrostatic map\"):\n    \"\"\"Which level of theory answers the question at an affordable cost.\"\"\"\n    needs_electrons = want in (\"partial charges and an electrostatic map\",\n                               \"heat of formation\",\n                               \"transition state of a reaction\")\n    if not needs_electrons:\n        rec = \"Molecular mechanics\"\n        why = (\"The question is about geometry and sterics only. A force field \"\n               \"treats atoms as spheres on springs and never computes an \"\n               \"electron distribution, which is all the speed comes from.\")\n    elif atoms <= 30:\n        rec = \"Ab initio quantum mechanics\"\n        why = (\"Electron distribution is required and the molecule is small \"\n               \"enough that a first-principles calculation is affordable.\")\n    elif atoms <= 150:\n        rec = \"Semi-empirical quantum mechanics\"\n        why = (\"Electron distribution is required but the molecule is too large \"\n               \"for ab initio, so experimentally fitted parameters are used to \"\n               \"simplify the integrals.\")\n    else:\n        rec = \"Molecular mechanics, and reconsider the question\"\n        why = (\"No quantum method is affordable at this size. Either cut the \"\n               \"molecule down to the fragment that matters, or accept that \"\n               \"electronic properties are out of reach.\")\n    cost = {\"Molecular mechanics\": 1, \"Semi-empirical quantum mechanics\": 1e3,\n            \"Ab initio quantum mechanics\": 1e6}\n    f, ax = _fig(8.6, 2.9)\n    names = list(cost); vals = [cost[n] for n in names]\n    cols = [PRIMARY if n in rec else MUTED for n in names]\n    ax.barh(names, vals, color=cols)\n    ax.set_xscale(\"log\"); ax.set_xlabel(\"relative cost (log scale)\")\n    ax.set_title(f\"{atoms} atoms   -   {want}\")\n    plt.tight_layout(); plt.show()\n    print(\"RECOMMEND:\", rec); print()\n    print(_tw(why))\n    return dict(atoms=atoms, question=want, recommend=rec,\n                needs_electron_distribution=needs_electrons)\n\n\n# ---------------------------------------------------------------- L25 extra\ndef site_surface(probe_radius=1.4, show_probe=True):\n    \"\"\"van der Waals surface versus the solvent-accessible (Connolly) surface.\n\n    Two gate atoms are placed so the opening between their van der Waals\n    surfaces is exactly GAP angstroms. A spherical probe passes only if its\n    radius does not exceed GAP/2, so the narration below is derived from the\n    coordinates rather than asserted.\n    \"\"\"\n    R = 1.7\n    GAP = 1.60                       # surface-to-surface opening, angstroms\n    d = 2 * R + GAP                  # required centre-to-centre separation\n    gate = [(-d / 2, 0.0, R), (d / 2, 0.0, R)]\n    wall = [(-5.6, -1.0, R), (-5.0, -3.0, R), (-3.4, -4.4, R),\n            (-1.2, -5.0, R), (1.2, -5.0, R), (3.4, -4.4, R),\n            (5.0, -3.0, R), (5.6, -1.0, R)]\n    atoms = gate + wall\n    half_gap = GAP / 2\n    fits = probe_radius <= half_gap\n\n    gx, gy = np.meshgrid(np.linspace(-8, 8, 460), np.linspace(-7.5, 4.5, 360))\n    vdw = np.zeros_like(gx, bool); excluded = np.zeros_like(gx, bool)\n    for (cx, cy, r) in atoms:\n        dd = (gx - cx) ** 2 + (gy - cy) ** 2\n        vdw |= dd <= r ** 2\n        excluded |= dd <= (r + probe_radius) ** 2\n\n    f, ax = _fig(7.4, 6.0)\n    ax.contourf(gx, gy, excluded.astype(float), levels=[.5, 1.5],\n                colors=[\"#CBD5E1\"], alpha=.8)\n    ax.contour(gx, gy, excluded.astype(float), levels=[.5], colors=[SECOND],\n               linewidths=2.2)\n    ax.contourf(gx, gy, vdw.astype(float), levels=[.5, 1.5], colors=[MUTED],\n                alpha=.95)\n    ax.contour(gx, gy, vdw.astype(float), levels=[.5], colors=[\"#334155\"],\n               linewidths=1.3)\n    if show_probe:\n        ax.add_patch(plt.Circle((0.0, 0.0), probe_radius, fill=False,\n                                ec=PRIMARY, lw=2.4, ls=\"--\"))\n        ax.plot([0], [0], \"x\", color=PRIMARY, ms=9)\n    ax.annotate(\"\", xy=(-d / 2 + R, 0), xytext=(d / 2 - R, 0),\n                arrowprops=dict(arrowstyle=\"<->\", color=PRIMARY, lw=1.6))\n    ax.text(0, .45, f\"{GAP:.2f} A opening\", ha=\"center\", color=PRIMARY,\n            fontsize=9)\n    ax.set_aspect(\"equal\"); ax.set_xlim(-8, 8); ax.set_ylim(-7.5, 4.5)\n    ax.set_xticks([]); ax.set_yticks([])\n    ax.set_title(f\"probe radius {probe_radius:.2f} A\\n\"\n                 \"dark grey = atoms (van der Waals)    \"\n                 \"orange outline = surface this probe can reach\")\n    plt.tight_layout(); plt.show()\n\n    print(f\"opening between the gate atoms : {GAP:.2f} A\")\n    print(f\"largest probe that passes      : {half_gap:.2f} A radius\")\n    print(f\"this probe                     : {probe_radius:.2f} A radius\")\n    print()\n    print(_tw(\n        (\"The probe fits through the opening, so the cavity behind the gate is \"\n         \"mapped as accessible and a ligand atom may be placed there.\"\n         if fits else\n         \"The probe is too large to pass, so the opening is sealed over. The \"\n         \"cavity behind it never appears on the docking surface, which is the \"\n         \"point: no ligand atom could reach it either.\")))\n    return dict(probe_radius=probe_radius, opening_angstrom=GAP,\n                max_probe_radius_that_fits=half_gap,\n                probe_passes=bool(fits),\n                excluded_volume_grid_points=int(excluded.sum()))\n\n\n# ------------------------------------------------- L31, real SwissBioisostere\ndef bioisostere_real(parent=\"carboxylic acid\", replacement=\"primary amide\",\n                     need_not_worse=0.80, max_mw_increase=60.0, show=True):\n    \"\"\"Judge a replacement against real matched-pair data.\n\n    Data: SwissBioisostere (CC-BY 4.0), ChEMBL 36. See swissbio.py for the\n    full citation and for what was changed on retrieval.\n    \"\"\"\n    import swissbio as sb\n    rows = {r[\"name\"]: r for r in sb.table(parent)}\n    if replacement not in rows:\n        print(f\"'{replacement}' is not in the curated {parent} list.\")\n        print(\"Available:\", \", \".join(sorted(rows)[:12]), \"...\")\n        return None\n    r = rows[replacement]\n    n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n    not_worse = (r[\"better\"] + r[\"similar\"]) / n\n    total = sb.QUERIES[parent][\"total\"]\n\n    if show:\n        f, (a1, a2) = plt.subplots(1, 2, figsize=(12.6, 4.2))\n        # how activity changed across the real pairs\n        a1.barh([\"worse\", \"similar\", \"better\"],\n                [r[\"worse\"], r[\"similar\"], r[\"better\"]],\n                color=[SECOND, MUTED, \"#15803D\"])\n        a1.set_xlabel(f\"matched molecular pairs (n = {n})\")\n        a1.set_title(f\"{replacement}\\nreplacing {parent} in real compounds\")\n        for s in (\"top\", \"right\"): a1.spines[s].set_visible(False)\n\n        # where this replacement sits against the others by frequency\n        tab = sb.table(parent)\n        ranks = [x[\"rank\"] for x in tab]; freqs = [x[\"freq\"] for x in tab]\n        cols = [PRIMARY if x[\"name\"] == replacement else \"#CBD5E1\" for x in tab]\n        a2.bar(ranks, freqs, color=cols)\n        a2.set_xlabel(f\"rank by frequency (top {len(tab)} of {total:,})\")\n        a2.set_ylabel(\"matched pairs\")\n        a2.set_title(f\"this replacement is rank {r['rank']} of {total:,}\")\n        for s in (\"top\", \"right\"): a2.spines[s].set_visible(False)\n        plt.tight_layout(); plt.show()\n\n    print(f\"SMILES              {r['smiles']}\")\n    print(f\"frequency           {r['freq']:,} matched pairs  (rank {r['rank']} of {total:,})\")\n    print(f\"activity outcome    {r['better']} better / {r['similar']} similar / {r['worse']} worse\")\n    print(f\"not worse           {not_worse*100:.0f}% of pairs\")\n    print(f\"mean change         logP {r['dlogp']:+.2f}   tPSA {r['dtpsa']:+.1f}   MW {r['dmw']:+.1f}\")\n    print()\n    checks = [(f\"at least {need_not_worse*100:.0f}% of pairs not worse\", not_worse >= need_not_worse),\n              (f\"molecular weight increase under {max_mw_increase:.0f} Da\",\n               r[\"dmw\"] < max_mw_increase)]\n    for text, okd in checks:\n        print((\"  MET      \" if okd else \"  NOT MET  \") + text)\n    return dict(name=replacement, smiles=r[\"smiles\"], freq=r[\"freq\"], rank=r[\"rank\"],\n                not_worse=round(not_worse, 3), dlogp=r[\"dlogp\"], dtpsa=r[\"dtpsa\"],\n                dmw=r[\"dmw\"], met=sum(1 for _, o in checks if o), total_checks=len(checks))\n\n\ndef bioisostere_ranking(parent=\"carboxylic acid\", min_freq=300, show=True):\n    \"\"\"Frequency is not success: rank the common replacements by outcome.\"\"\"\n    import swissbio as sb\n    tab = [r for r in sb.table(parent) if r[\"freq\"] >= min_freq]\n    for r in tab:\n        n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n        r[\"not_worse\"] = (r[\"better\"] + r[\"similar\"]) / n if n else 0\n    by_freq = sorted(tab, key=lambda r: -r[\"freq\"])\n    by_ok = sorted(tab, key=lambda r: -r[\"not_worse\"])\n    if show:\n        f, ax = _fig(9.6, 5.0)\n        ax.scatter([r[\"freq\"] for r in tab], [r[\"not_worse\"] * 100 for r in tab],\n                   s=70, color=PRIMARY, zorder=3)\n        for r in tab:\n            if r[\"freq\"] > 600 or r[\"not_worse\"] > 0.88:\n                ax.annotate(r[\"name\"], (r[\"freq\"], r[\"not_worse\"] * 100),\n                            fontsize=8, xytext=(5, 4), textcoords=\"offset points\")\n        ax.set_xscale(\"log\")\n        ax.set_xlabel(f\"how often it was done (matched pairs, log scale)\")\n        ax.set_ylabel(\"% of pairs where activity was not worse\")\n        ax.set_title(f\"replacing {parent}: popularity against outcome\")\n        plt.tight_layout(); plt.show()\n    print(f\"{'most common':<30}{'most reliable':<30}\")\n    print(\"-\" * 60)\n    for i in range(min(6, len(tab))):\n        print(f\"{by_freq[i]['name'][:28]:<30}{by_ok[i]['name'][:28]:<30}\")\n    return dict(n=len(tab),\n                most_common=by_freq[0][\"name\"], most_reliable=by_ok[0][\"name\"],\n                same=by_freq[0][\"name\"] == by_ok[0][\"name\"])\n", "sims.py", "exec"), _sm.__dict__)
sys.modules["sims"] = _sm
sims = _sm

ITEMS = [
    Item("P1", "Biological activity is entered into a QSAR equation as log(1/C). Why the reciprocal?",
         {"A": "So that negative activity values can never be generated", "B": "So that the units of concentration cancel in the equation", "C": "So that the scale rises as the concentration needed falls", "D": "So that the equation becomes linear in every descriptor"},
         "C", {"B": "Taking a reciprocal does not cancel units; the logarithm handles the scale.", "D": "Linearity depends on the chemistry, not on how activity is expressed.", "A": "Negative values of log(1/C) are perfectly possible and interpretable.", "_correct": "A smaller concentration means a more potent compound, so the reciprocal makes the numerical scale increase with potency."},
         bloom="Understand", silo="", section="precheck"),
    Item("P2", "A substituent has pi = 1.68. What does that positive value indicate?",
         {"A": "It is more hydrophobic than a hydrogen at the same position", "B": "It is more electron-withdrawing than a hydrogen at that site", "C": "It is larger in van der Waals volume than a hydrogen atom", "D": "It is more readily metabolized than a hydrogen at that site"},
         "A", {"B": "Electronic effects are described by sigma, not by pi.", "C": "Size is described by molar refractivity or a steric parameter, not by pi.", "D": "Metabolic liability is not what any substituent constant measures.", "_correct": "The parameter pi is defined as the change in log P on replacing hydrogen, so a positive value means greater hydrophobicity."},
         bloom="Understand", silo="", section="precheck"),
    Item("P3", "An equation contains a negative coefficient on a squared log P term. What does the curve look like?",
         {"A": "It rises steadily with log P across the whole range studied", "B": "It is flat, since the squared term cancels the linear term", "C": "It falls to a minimum at an optimum log P and then increases", "D": "It rises to a maximum at an optimum log P and then declines"},
         "D", {"A": "A steady rise would need no squared term at all.", "C": "A negative coefficient on the squared term gives a maximum, not a minimum.", "B": "The two terms do not cancel; together they describe a curve.", "_correct": "A negative coefficient on the squared term makes the parabola open downward, so activity peaks at an optimum log P."},
         bloom="Understand", silo="", section="precheck"),
    Item("C1", "Fitting pi as a straight line gave r = 0.48; as a parabola it gave r = 0.93. What does that difference establish?",
         {"A": "Hydrophobicity matters, but the relationship is not a straight line", "B": "Hydrophobicity does not matter, since the linear fit was so poor", "C": "The parabola has over-fitted, since adding a term always helps r", "D": "The activity data for these eleven analogues must be unreliable"},
         "A", {"B": "A fit of 0.93 shows hydrophobicity matters a great deal in this series.", "C": "Adding a term can only help r slightly by chance; a jump of 0.45 is structural.", "D": "The data support a tight curve, which is evidence of reliability, not error.", "_correct": "The strong quadratic fit shows a real dependence on hydrophobicity that a straight line could not capture, with activity peaking at an optimum."},
         bloom="Analyze", silo="3", section="concept"),
    Item("C2", "The van der Waals volume descriptor gave r near 0.01 linear and 0.12 quadratic. What is the sound conclusion?",
         {"A": "Steric bulk does not govern activity across this particular series", "B": "Steric bulk governs activity but needs a cubic term to be seen", "C": "The volume values must have been tabulated for the wrong atoms", "D": "Steric bulk matters only once hydrophobicity has been removed"},
         "A", {"B": "Adding higher powers to a descriptor with no signal fits noise rather than chemistry.", "C": "There is no evidence of a tabulation error; the descriptor simply does not correlate.", "D": "Nothing in this fit suggests a suppressed effect waiting to emerge.", "_correct": "Neither functional form extracts any signal, so within this series variation in steric bulk is not what drives the differences in activity."},
         bloom="Evaluate", silo="6", section="concept"),
    Item("Q1", "A substituent is reported with pi = 1.68 and sigma = -0.20. What does this pair describe?",
         {"A": "Less hydrophobic than hydrogen and strongly electron-accepting", "B": "More hydrophobic than hydrogen and weakly electron-donating", "C": "More hydrophobic than hydrogen and strongly electron-accepting", "D": "Less hydrophobic than hydrogen and weakly electron-releasing"},
         "B", {"C": "The hydrophobicity sign is correct here, but the electronic sign is inverted", "D": "The electronic sign is correct here, but the hydrophobicity sign is inverted", "A": "This inverts both conventions, as the group is more hydrophobic and electron-donating", "_correct": "A positive pi means more hydrophobic than hydrogen, and a small negative sigma means weakly electron-donating."},
         bloom="Understand", silo="3", section="post"),
    Item("Q2", "Why is biological activity entered into a QSAR equation as log(1/C)?",
         {"A": "Because concentration terms must be dimensionless before regression", "B": "Because the reciprocal converts an EC50 value into a Ki value", "C": "Because logarithms remove the need to report standard deviations", "D": "Because potency spans orders of magnitude and rises as C falls"},
         "D", {"A": "is an invented requirement", "C": "is false", "B": "confuses two different parameters", "_correct": "The reciprocal makes the scale rise with potency, and the logarithm compresses concentrations that span several orders of magnitude."},
         bloom="Understand", silo="3", section="post"),
    Item("Q3", "A series of ethers fits log(1/C) = -0.22(log P)^2 + 1.04 log P + 2.16. What does the negative squared term indicate?",
         {"A": "The data were fitted with too few compounds to be reliable", "B": "Activity falls steadily as hydrophobicity increases throughout", "C": "Activity peaks at an optimum hydrophobicity and then declines", "D": "Hydrophobicity makes no measurable contribution to the activity"},
         "C", {"A": "is a statistical worry, not the meaning of the term", "B": "describes a negative linear term", "D": "is contradicted by the equation", "_correct": "A negative coefficient on the squared term makes the curve concave, so activity rises to an optimum log P and then falls away."},
         bloom="Understand", silo="3", section="post"),
    Item("Q4", "Hammett sigma values are tabulated for meta and para substituents but not ortho. Why?",
         {"A": "Ortho substituents cannot transmit inductive effects to the acid", "B": "Steric interference at ortho positions corrupts the measurement", "C": "Ortho positions are not conjugated with the carboxylate group", "D": "Ortho-substituted benzoic acids are too insoluble to titrate"},
         "B", {"A": "Ortho substituents do transmit inductive effects, so this claim is electronically false", "D": "Insolubility sounds plausible but is not why ortho substituents are excluded", "C": "Ortho positions are conjugated with the ring and the carboxylate, so this is false", "_correct": "An ortho substituent sits next to the reacting group, so its steric effect contaminates the purely electronic quantity sigma is meant to isolate."},
         bloom="Understand", silo="3", section="post"),
    Item("Q5", "Taft's Es for hydrogen is 0.00 and for tert-butyl is -2.78. What does the more negative value mean?",
         {"A": "The group donates electron density more strongly than hydrogen", "B": "The group hinders the reference reaction more than hydrogen does", "C": "The group lowers the molar refractivity of the parent compound", "D": "The group reduces the compound's aqueous solubility more sharply"},
         "B", {"C": "Molar refractivity is a separate parameter and is not what this term reports", "D": "Aqueous solubility is an unrelated property that this parameter does not capture", "A": "Electron donation is an electronic effect, while this parameter measures sterics", "_correct": "Es is defined from the rate of a reference reaction, and a more negative value means the group hinders that reaction more than hydrogen does."},
         bloom="Understand", silo="3", section="post"),
]

gb = Gradebook("L29", "Quantitative Structure\u2013Activity Relationships I")
gb.add_many(ITEMS)
print("Ready - 10 questions loaded. Scroll down and begin.")

# ===== BEGIN structure-drawing and calculator kit (lecture notes) =====
# ---------------------------------------------------------------------------
# PHS 511 notebook helper library (pure Python + matplotlib; no RDKit, no network)
# Draws textbook-oriented 2D skeletal structures from explicit coordinates so
# every figure has a fixed, reviewed orientation:
#   * acyl chains: C1 (carbonyl) on the LEFT, omega (methyl) end on the RIGHT
#   * glycerol / sphingosine backbones vertical (sn-1 top, sn-3 bottom)
#   * steroids: rings A-B-C-D left to right, C3-OH lower left, C18/C19 up
# ---------------------------------------------------------------------------
import math
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyArrowPatch, FancyBboxPatch, Circle

COL = {
    "ink": "#1f2430", "muted": "#6b7280",
    "ester": "#1d6fb8", "amide": "#c2410c", "ether": "#7c3aed",
    "phos": "#15803d", "oh": "#b91c1c", "dbl": "#d97706",
    "delta": "#1d6fb8", "omega": "#b91c1c", "drug": "#be123c",
    "polar": "#0e7490", "nonpolar": "#a16207", "hl": "#fde68a",
}
S30, C30 = 0.5, math.cos(math.radians(30))
UNIT_IN = 0.36   # inches per bond length -> identical scale in every figure


class Mol:
    """Atoms with explicit 2D coordinates; bonds with order/style/highlight."""

    def __init__(self, name=""):
        self.name, self.atoms, self.bonds, self.notes, self.boxes = name, [], [], [], []

    def add(self, x, y, label="", color=None, ha="center", size=None, anchor=None):
        """anchor: index of the label character that sits on the atom (default: first C/N/O/P/S)."""
        self.atoms.append(dict(x=x, y=y, label=label, color=color, ha=ha, size=size, anchor=anchor))
        return len(self.atoms) - 1

    def bond(self, i, j, order=1, style="plain", hl=None, color=None, side=None):
        self.bonds.append(dict(i=i, j=j, order=order, style=style, hl=hl, color=color, side=side))
        return len(self.bonds) - 1

    def note(self, x, y, text, color=None, size=8.5, ha="center", va="center", weight="normal"):
        self.notes.append(dict(x=x, y=y, text=text, color=color, size=size, ha=ha, va=va, weight=weight))

    def pos(self, i):
        a = self.atoms[i]
        return a["x"], a["y"]



def _turtle(m, start, n, d0, turns, orders, hl=None):
    idx = [start]
    x, y = m.pos(start)
    d = d0
    for k in range(1, n + 1):
        x, y = x + math.cos(math.radians(d)), y + math.sin(math.radians(d))
        idx.append(m.add(x, y))
        m.bond(idx[-2], idx[-1], order=orders.get(k, 1), hl=hl)
        if k < n:
            d += turns[k]
    return idx


def chain(m, start, n, heading=0.0, first=+1, cis=(), trans=(), hl=None, flip=()):
    """Zigzag chain of n carbons grown from atom `start` (start = carbon 0).
    Bond k joins chain carbons k-1 and k. The turn at carbon k normally
    alternates sign (zigzag). If bond k is a cis C=C, the turn at carbon k
    repeats the previous sign so both substituents sit on the same side.
    `flip` lists carbons where a single bond is drawn rotated (same-sign turn)
    to steer a long chain back into its row; it does not change the molecule."""
    turns, orders = {}, {}
    sign = -first
    for k in range(1, n):
        if k >= 2:
            sign = sign if (k in cis or k in flip) else -sign
        turns[k] = 60 * sign
    for k in list(cis) + list(trans):
        orders[k] = 2
    return _turtle(m, start, n, heading + 30 * first, turns, orders, hl)


# ---------------------------------------------------------------------------
# Rendering
# ---------------------------------------------------------------------------
SUBS = set("\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089\u207a\u207b\u00b9\u00b2\u00b3\u2074")
EM = 1.0 / (72 * UNIT_IN)   # data units per point at the fixed drawing scale


def _cw(ch):
    """Approximate glyph advance (em) for DejaVu Sans."""
    if ch in SUBS:
        return 0.40
    if ch in "()":
        return 0.39
    if ch.isupper():
        return 0.72
    if ch.isdigit():
        return 0.64
    return 0.56


def _label_box(lab, fs, anchor=None):
    """(left, right, half-height) of a label in data units, measured from the
    anchor glyph centre (first C/N/O/P/S), which sits exactly on the atom."""
    em = fs * EM
    ws = [_cw(c) * em for c in lab]
    k = anchor if anchor is not None else next((i for i, c in enumerate(lab) if c in "CNOPS"), 0)
    left = sum(ws[:k]) + ws[k] / 2
    right = sum(ws[k + 1:]) + ws[k] / 2
    return left, right, 0.36 * fs * EM * 1.0


def _label_radius(lab, ux, uy, fs=10.5, anchor=None):
    """Distance from atom centre to the label box edge along (ux, uy), plus a gap."""
    if not lab:
        return 0.0
    left, right, hh = _label_box(lab, fs, anchor)
    tx = (right if ux > 0 else left) / abs(ux) if abs(ux) > 1e-9 else 1e9
    ty = hh / abs(uy) if abs(uy) > 1e-9 else 1e9
    return min(tx, ty) + 0.08


def draw(ax, m, fs=10.5):
    A = m.atoms
    nb = {i: [] for i in range(len(A))}
    for b in m.bonds:
        nb[b["i"]].append(b["j"]); nb[b["j"]].append(b["i"])
    # highlights first (under everything)
    for b in m.bonds:
        if b["hl"]:
            (x1, y1), (x2, y2) = m.pos(b["i"]), m.pos(b["j"])
            ax.plot([x1, x2], [y1, y2], color=b["hl"], lw=9, alpha=0.30,
                    solid_capstyle="round", zorder=1)
    for b in m.bonds:
        i, j = b["i"], b["j"]
        (x1, y1), (x2, y2) = m.pos(i), m.pos(j)
        L = math.hypot(x2 - x1, y2 - y1)
        ux, uy = (x2 - x1) / L, (y2 - y1) / L
        r1 = _label_radius(A[i]["label"], ux, uy, A[i]["size"] or fs, A[i]["anchor"])
        r2 = _label_radius(A[j]["label"], ux, uy, A[j]["size"] or fs, A[j]["anchor"])
        ax1, ay1 = x1 + ux * r1, y1 + uy * r1
        ax2, ay2 = x2 - ux * r2, y2 - uy * r2
        c = b["color"] or COL["ink"]
        st = b["style"]
        if st == "wedge":
            px, py = -uy * 0.11, ux * 0.11
            ax.add_patch(Polygon([(ax1, ay1), (ax2 + px, ay2 + py), (ax2 - px, ay2 - py)],
                                 closed=True, fc=c, ec=c, lw=0.6, zorder=3))
            continue
        if st == "hash":
            nseg = 7
            for s in range(nseg + 1):
                f = s / nseg
                cx, cy = ax1 + (ax2 - ax1) * f, ay1 + (ay2 - ay1) * f
                w = 0.02 + 0.10 * f
                ax.plot([cx - uy * w, cx + uy * w], [cy + ux * w, cy - ux * w],
                        color=c, lw=1.1, zorder=3, solid_capstyle="butt")
            continue
        if st == "wavy":
            n = 40
            xs = [ax1 + (ax2 - ax1) * k / n - uy * 0.06 * math.sin(k / n * 6 * math.pi) for k in range(n + 1)]
            ys = [ay1 + (ay2 - ay1) * k / n + ux * 0.06 * math.sin(k / n * 6 * math.pi) for k in range(n + 1)]
            ax.plot(xs, ys, color=c, lw=1.4, zorder=3)
            continue
        px, py = -uy, ux
        if b["order"] == 1:
            ax.plot([ax1, ax2], [ay1, ay2], color=c, lw=1.6, zorder=3, solid_capstyle="round",
                    ls="--" if st == "dashed" else "-")
            continue
        side = b["side"]
        if side is None:
            if A[i]["label"] or A[j]["label"]:
                side = 0                      # C=O, C=N+: centred pair
            else:
                sx = sy = 0.0
                for k in nb[i]:
                    if k != j:
                        sx += A[k]["x"] - x1; sy += A[k]["y"] - y1
                for k in nb[j]:
                    if k != i:
                        sx += A[k]["x"] - x2; sy += A[k]["y"] - y2
                cross = ux * sy - uy * sx
                side = 1 if cross >= -1e-6 else -1
        if side == 0:
            for sgn in (+1, -1):
                o = 0.075 * sgn
                ax.plot([ax1 + px * o, ax2 + px * o], [ay1 + py * o, ay2 + py * o],
                        color=c, lw=1.5, zorder=3, solid_capstyle="round")
        else:
            ax.plot([ax1, ax2], [ay1, ay2], color=c, lw=1.6, zorder=3, solid_capstyle="round")
            o, sh = 0.15 * side, 0.16
            ax.plot([ax1 + px * o + ux * sh, ax2 + px * o - ux * sh],
                    [ay1 + py * o + uy * sh, ay2 + py * o - uy * sh],
                    color=c, lw=1.5, zorder=3, solid_capstyle="round")
    for a in A:
        if a["label"]:
            l_, r_, _ = _label_box(a["label"], a["size"] or fs, a["anchor"])
            ax.text(a["x"] + (r_ - l_) / 2, a["y"], a["label"], ha="center", va="center",
                    fontsize=a["size"] or fs, color=a["color"] or COL["ink"],
                    zorder=5, fontweight="normal")
    for n_ in m.notes:
        ax.text(n_["x"], n_["y"], n_["text"], ha=n_["ha"], va=n_["va"], fontsize=n_["size"],
                color=n_["color"] or COL["muted"], zorder=6, fontweight=n_["weight"])


def extent(m, pad=0.7):
    xs = [a["x"] for a in m.atoms] + [n["x"] for n in m.notes]
    ys = [a["y"] for a in m.atoms] + [n["y"] for n in m.notes]
    return min(xs) - pad, max(xs) + pad, min(ys) - pad, max(ys) + pad




def show(m, title=None, unit=UNIT_IN, extra=None, caption=None, show_now=True):
    """One structure at the fixed scale (UNIT_IN inches per bond)."""
    x0, x1, y0, y1 = extent(m)
    if extra and hasattr(extra, "bounds"):
        bx0, bx1, by0, by1 = extra.bounds
        x0, x1, y0, y1 = min(x0, bx0), max(x1, bx1), min(y0, by0), max(y1, by1)
    th = 0.42 if title else 0.0
    ch = 0.40 if caption else 0.0
    W, H = (x1 - x0) * unit, (y1 - y0) * unit
    fig = plt.figure(figsize=(W, H + th + ch))
    ax = fig.add_axes([0, ch / (H + th + ch), 1, H / (H + th + ch)])
    draw(ax, m)
    if extra:
        extra(ax)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        fig.text(0.0, 1 - (th / 2) / (H + th + ch), title, fontsize=11.5, fontweight="bold",
                 color=COL["ink"], va="center", ha="left")
    if caption:
        fig.text(0.0, (ch / 2) / (H + th + ch), caption, fontsize=9, color=COL["muted"],
                 va="center", ha="left")
    if show_now:
        plt.show()
    return fig


def grid(mols, titles, ncols=2, unit=UNIT_IN, align="left", suptitle=None, captions=None, show_now=True, bottom=0.0):
    """Panels drawn at one common scale; each panel box has the same width and
    the row height of its tallest member, so structures line up cleanly."""
    exts = [extent(m) for m in mols]
    W = max(max(e[1] - e[0] for e in exts), 3.6 / unit)   # panels at least 3.6 in wide
    nrows = math.ceil(len(mols) / ncols)
    import textwrap as _tw
    maxch = max(12, int(W * unit * 72 / (10.5 * 0.60)) - 1)
    wrapped = [_tw.fill(t, maxch) for t in titles]
    rowH = []
    for r in range(nrows):
        rowH.append(max(exts[k][3] - exts[k][2] for k in range(r * ncols, min(len(mols), (r + 1) * ncols))))
    row_th = []
    for r in range(nrows):
        nl = max(wrapped[k].count("\n") + 1 for k in range(r * ncols, min(len(mols), (r + 1) * ncols)))
        row_th.append(0.27 + 0.19 * nl)
    fig_w = W * unit * ncols
    fig_h = sum(h * unit + th for h, th in zip(rowH, row_th)) + (0.5 if suptitle else 0) + bottom
    fig = plt.figure(figsize=(fig_w, fig_h))
    top = 1.0 - (0.5 / fig_h if suptitle else 0)
    y_cursor = top
    for r in range(nrows):
        h_frac = (rowH[r] * unit) / fig_h
        t_frac = row_th[r] / fig_h
        y_cursor -= (h_frac + t_frac)
        for c in range(ncols):
            k = r * ncols + c
            if k >= len(mols):
                break
            ax = fig.add_axes([c / ncols, y_cursor, 1 / ncols, h_frac])
            m = mols[k]
            x0, x1, y0, y1 = exts[k]
            draw(ax, m)
            if align == "left":
                ax.set_xlim(x0, x0 + W)
            else:
                cx = (x0 + x1) / 2
                ax.set_xlim(cx - W / 2, cx + W / 2)
            cy = (y0 + y1) / 2
            ax.set_ylim(cy - rowH[r] / 2, cy + rowH[r] / 2)
            ax.set_aspect("equal"); ax.axis("off")
            ax.text(0.01, 1.0 + 0.02, wrapped[k], transform=ax.transAxes, fontsize=10.5, linespacing=1.15,
                    fontweight="bold", color=COL["ink"], va="bottom", ha="left")
    if suptitle:
        fig.text(0.005, 1 - 0.25 / fig_h, suptitle, fontsize=12.5, fontweight="bold", va="center")
    if show_now:
        plt.show()
    return fig


# ---------------------------------------------------------------------------
# Structure builders (coordinates chosen to match textbook orientation)
# ---------------------------------------------------------------------------
def _outer(m, i, nbrs, dist=0.55):
    """Point on the open (outer) side of atom i, away from its neighbours."""
    x, y = m.pos(i)
    sx = sum(m.atoms[k]["x"] - x for k in nbrs); sy = sum(m.atoms[k]["y"] - y for k in nbrs)
    L = math.hypot(sx, sy) or 1.0
    return x - sx / L * dist, y - sy / L * dist


def fatty_acid(n, cis=(), trans=(), name="", numbers=None, ionized=False, origin=(0.0, 0.0),
               hl_double=True, flip=()):
    """Free fatty acid, C1 (COOH) on the LEFT, methyl (omega) end on the RIGHT.
    cis/trans: Delta positions (bond between C_p and C_p+1).
    numbers: None | 'delta' | 'omega' | 'both'"""
    m = Mol(name)
    ox, oy = origin
    c1 = m.add(ox, oy)
    o_dbl = m.add(ox, oy + 1.0, "O")
    o_h = m.add(ox - C30, oy - 0.5, "⁻O" if ionized else "HO")
    m.bond(c1, o_dbl, 2); m.bond(c1, o_h, 1)
    hl = COL["hl"] if hl_double else None
    ids = chain(m, c1, n - 1, heading=0, first=-1, cis=cis, trans=trans, flip=flip)
    if hl_double:
        for b in m.bonds:
            if b["order"] == 2 and not m.atoms[b["i"]]["label"] and not m.atoms[b["j"]]["label"]:
                b["hl"] = COL["dbl"]
    m.carbons = ids  # carbons[0] = C1
    if numbers:
        dbs = sorted(list(cis) + list(trans))
        last = dbs[-1] if dbs else n - 2
        for p, a in enumerate(ids, start=1):
            if p == 1:
                nb = [o_dbl, o_h, ids[1]]
            elif p == n:
                nb = [ids[p - 2]]
            else:
                nb = [ids[p - 2], ids[p]]
            if numbers in ("delta", "both"):
                x, y = _outer(m, a, nb, 0.50)
                m.note(x, y, str(p), color=COL["delta"], size=7.5)
            w = n - p + 1
            if numbers == "omega" or (numbers == "both" and p >= last):
                if p == n:
                    x, y = _outer(m, a, nb, 0.50)
                    y -= 0.42 if numbers == "both" else 0.0
                else:
                    x, y = _outer(m, a, nb, 0.50 if numbers == "omega" else -0.62)
                m.note(x, y, ("\u03c9" + str(w)) if w == 1 else str(w), color=COL["omega"], size=7.5,
                       weight="bold")
    return m


def acyl_row(m, o_atom, n, cis=(), trans=(), carbonyl_up=True, flip=None):
    """Acyl group C(=O)-chain attached to ester/amide atom o_atom, chain to the right."""
    x, y = m.pos(o_atom)
    c1 = m.add(x + C30, y + (0.5 if carbonyl_up else -0.5))
    m.bond(o_atom, c1, 1)
    od = m.add(x + C30, y + (1.5 if carbonyl_up else -1.5), "O")
    m.bond(c1, od, 2)
    if flip is None:   # keep long chains in their row: rotate the bond 3 carbons past the last cis C=C
        flip = tuple(p + 2 for p in cis)
    ids = chain(m, c1, n - 1, heading=0, first=-1 if carbonyl_up else +1, cis=cis, trans=trans, flip=flip)
    for b in m.bonds:
        if b["order"] == 2 and not m.atoms[b["i"]]["label"] and not m.atoms[b["j"]]["label"]:
            b["hl"] = COL["dbl"]
    return c1, od, ids


ROW = 2.6  # vertical spacing of backbone carbons in E-shaped lipids


def backbone(m, labels, x=0.0, y0=0.0):
    """Vertical 3-carbon backbone (glycerol or sphingosine C1-C3), top->bottom."""
    ids = []
    for k, lab in enumerate(labels):
        ids.append(m.add(x, y0 - k * ROW, lab))
        if k:
            m.bond(ids[k - 1], ids[k], 1)
    return ids


def ester(m, c_backbone, n, cis=(), trans=(), hl=True, cut=None):
    x, y = m.pos(c_backbone)
    o = m.add(x + 1.15, y, "O")
    b0 = m.bond(c_backbone, o, 1, hl=COL["ester"] if hl else None)
    c1, od, ids = acyl_row(m, o, n, cis, trans)
    m.bonds[-(len(ids)) - 1]["hl"] = None
    # highlight O-C(=O) (the ester bond lipases cut)
    for b in m.bonds:
        if b["i"] == o and b["j"] == c1:
            b["hl"] = COL["ester"] if hl else None
            if cut:
                b["color"] = COL["drug"]
    return o, c1, ids


def triacylglycerol(acyls=((16, ()), (18, (9,)), (18, (9, 12))), name="Triacylglycerol",
                    sn=True, cuts=None):
    """E-shaped TAG; glycerol vertical, sn-1 top. acyls: ((n, cis-positions), ...)."""
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    rows = []
    for k, (n, cis) in enumerate(acyls):
        rows.append(ester(m, g[k], n, cis=cis))
    if sn:
        for k in range(3):
            x, y = m.pos(g[k])
            m.note(x - 0.95, y, "sn-%d" % (k + 1), color=COL["muted"], size=8.5, ha="right")
    m.g, m.rows = g, rows
    return m


def mark_cut(m, o_idx, c_idx, text, color=None, dy=-0.62, dx=0.0):
    x1, y1 = m.pos(o_idx); x2, y2 = m.pos(c_idx)
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    m.note(mx, my - 0.05, "✂", color=color or COL["drug"], size=13)
    m.note(mx + dx, my + dy, text, color=color or COL["drug"], size=7.5, ha="left" if dx else "center")


def phosphocholine(m, c_attach, head="choline"):
    """-O-P(=O)(O-)-O-CH2-CH2-N+(CH3)3 to the right of c_attach (horizontal)."""
    x, y = m.pos(c_attach)
    o1 = m.add(x + 1.15, y, "O"); m.bond(c_attach, o1, 1)
    p = m.add(x + 2.3, y, "P", color=COL["phos"]); m.bond(o1, p, 1, hl=COL["phos"])
    po = m.add(x + 2.3, y + 1.05, "O"); m.bond(p, po, 2)
    pm = m.add(x + 2.3, y - 1.05, "O⁻"); m.bond(p, pm, 1)
    o2 = m.add(x + 3.45, y, "O"); m.bond(p, o2, 1, hl=COL["phos"])
    if head == "choline":
        c_a = m.add(x + 3.45 + C30, y + 0.5); m.bond(o2, c_a)
        c_b = m.add(x + 3.45 + 2 * C30, y); m.bond(c_a, c_b)
        nn = m.add(x + 3.45 + 3 * C30, y + 0.5, "N⁺(CH₃)₃"); m.bond(c_b, nn)
        m.note(x + 3.45 + 1.6, y - 1.0, "choline", color=COL["muted"], size=8)
    elif head == "ethanolamine":
        c_a = m.add(x + 3.45 + C30, y + 0.5); m.bond(o2, c_a)
        c_b = m.add(x + 3.45 + 2 * C30, y); m.bond(c_a, c_b)
        nn = m.add(x + 3.45 + 3 * C30, y + 0.5, "NH₃⁺"); m.bond(c_b, nn)
        m.note(x + 3.45 + 1.4, y - 1.0, "ethanolamine", color=COL["muted"], size=8)
    return p


def phosphatidylcholine(name="Phosphatidylcholine (lecithin)", a1=(16, ()), a2=(18, (9,))):
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    ester(m, g[0], *a1); ester(m, g[1], *a2)
    phosphocholine(m, g[2], "choline")
    for k in range(3):
        x, y = m.pos(g[k]); m.note(x - 0.95, y, "sn-%d" % (k + 1), size=8.5, ha="right")
    m.g = g
    return m


def plasmalogen(name="Plasmalogen (ethanolamine)", n1=16, a2=(18, (9,))):
    """sn-1 vinyl ether: CH2-O-CH=CH-R (cis), no carbonyl."""
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    x, y = m.pos(g[0])
    o = m.add(x + 1.15, y, "O"); m.bond(g[0], o, 1, hl=COL["ether"])
    ids = chain(m, o, n1, heading=0, first=+1, cis=(2,), flip=(4,))
    # bond O-C1' is ether; C1'=C2' vinyl (cis, as in natural plasmalogens)
    for b in m.bonds:
        if b["i"] == o and b["j"] == ids[1]:
            b["hl"] = COL["ether"]
        if b["order"] == 2:
            b["hl"] = COL["ether"]
    ester(m, g[1], *a2)
    phosphocholine(m, g[2], "ethanolamine")
    for k in range(3):
        xx, yy = m.pos(g[k]); m.note(xx - 0.95, yy, "sn-%d" % (k + 1), size=8.5, ha="right")
    return m


def sphingo_core(m, fa_n=18, fa_cis=()):
    """Sphingosine drawn as the same vertical 3-carbon backbone:
    C3 top (CH-OH + trans CH=CH tail), C2 middle (CH-NH-acyl, amide), C1 bottom (CH2-O-head)."""
    s = backbone(m, ["CH", "CH", "CH₂"])
    x, y = m.pos(s[0])
    oh = m.add(x - 1.15, y, "HO"); m.bond(s[0], oh, 1, hl=COL["oh"])
    ids = chain(m, s[0], 15, heading=0, first=+1, trans=(1,))
    # C4=C5 trans: chain bond 1 is C3-C4? -> rebuild explicitly
    m.bonds[[k for k, b in enumerate(m.bonds) if b["i"] == s[0] and b["j"] == ids[1]][0]]["order"] = 1
    for b in m.bonds:
        if b["i"] == ids[1] and b["j"] == ids[2]:
            b["order"] = 2; b["hl"] = COL["dbl"]
    # amide at C2
    x2, y2 = m.pos(s[1])
    nh = m.add(x2 + 1.4, y2, "NH"); m.bond(s[1], nh, 1)
    c1, od, acyl = acyl_row(m, nh, fa_n, cis=fa_cis)
    for b in m.bonds:
        if b["i"] == nh and b["j"] == c1:
            b["hl"] = COL["amide"]
    for k, lab in enumerate(["C3", "C2", "C1"]):
        xx, yy = m.pos(s[k]); m.note(xx - 0.25, yy - 0.55, lab, size=7.5)
    return s


def sphingomyelin(name="Sphingomyelin"):
    m = Mol(name)
    s = sphingo_core(m, 18)
    phosphocholine(m, s[2], "choline")
    return m


def galactosylceramide(name="Galactosylceramide (a cerebroside)"):
    m = Mol(name)
    s = sphingo_core(m, 24, fa_cis=(15,))
    x, y = m.pos(s[2])
    o = m.add(x + 1.15, y, "O"); m.bond(s[2], o, 1, hl=COL["polar"])
    # beta-D-galactopyranose, Haworth-free chair-less hexagon (ring O upper right)
    cx, cy, r = x + 3.0, y - 0.1, 1.0
    ang = [180, 120, 60, 0, -60, -120]
    pts = [m.add(cx + r * math.cos(math.radians(a)), cy + r * math.sin(math.radians(a)),
                 "O" if k == 1 else "") for k, a in enumerate(ang)]
    for k in range(6):
        m.bond(pts[k], pts[(k + 1) % 6])
    m.bond(o, pts[0], 1, hl=COL["polar"])
    m.note(cx, cy - 0.05, "Gal", color=COL["polar"], size=9, weight="bold")
    return m


# --- steroids --------------------------------------------------------------
def _hex(cx, cy):
    return [(cx + math.cos(math.radians(a)), cy + math.sin(math.radians(a))) for a in (90, 30, -30, -90, -150, 150)]


def steroid_core(m, ox=0.0, oy=0.0, dbl=()):
    """Gonane nucleus in standard orientation. Returns dict C1..C17 -> atom index.
    Unit hexagon geometry with vertical shared A/B edge (C5-C10)."""
    r3 = math.sqrt(3)
    P = {}
    A = _hex(ox, oy)                      # top, ur, lr, bottom, ll, ul
    P.update({1: A[0], 10: A[1], 5: A[2], 4: A[3], 3: A[4], 2: A[5]})
    B = _hex(ox + r3, oy)
    P.update({9: B[0], 8: B[1], 6: B[3], 7: B[2]})
    C = _hex(ox + 1.5 * r3, oy + 1.5)
    P.update({12: C[0], 13: C[1], 14: C[2], 11: C[5]})
    # ring D: regular pentagon on edge C13-C14, outward (to the right)
    (x13, y13), (x14, y14) = P[13], P[14]
    mx, my = (x13 + x14) / 2, (y13 + y14) / 2
    s = 1.0
    apo = s / (2 * math.tan(math.radians(36)))
    cx, cy = mx + apo, my
    R = s / (2 * math.sin(math.radians(36)))
    base = math.degrees(math.atan2(y13 - cy, x13 - cx))
    pent = [(cx + R * math.cos(math.radians(base - 72 * k)), cy + R * math.sin(math.radians(base - 72 * k)))
            for k in range(5)]
    # pent[0]=C13, going clockwise: C17, C16, C15, C14
    P.update({17: pent[1], 16: pent[2], 15: pent[3]})
    idx = {k: m.add(v[0], v[1]) for k, v in P.items()}
    ring_bonds = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 10), (10, 1), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10),
                  (9, 11), (11, 12), (12, 13), (13, 14), (14, 8), (13, 17), (17, 16), (16, 15), (15, 14)]
    for a, b in ring_bonds:
        m.bond(idx[a], idx[b], 2 if (a, b) in dbl or (b, a) in dbl else 1)
    m.st = idx
    return idx


def subst(m, at, dx, dy, label="", style="plain", order=1, hl=None, color=None):
    x, y = m.pos(at)
    k = m.add(x + dx, y + dy, label, color=color)
    m.bond(at, k, order, style=style, hl=hl)
    return k


def side_chain_cholestane(m, st, extra_ethyl=False):
    """C20-C27 side chain on C17 (drawn up-right, zigzag)."""
    x, y = m.pos(st[17])
    c20 = m.add(x + 0.30, y + 0.95); m.bond(st[17], c20, 1, style="wedge")
    c21 = m.add(x - 0.45, y + 1.6); m.bond(c20, c21)
    ids = chain(m, c20, 4, heading=0, first=-1)
    c25 = ids[-1]
    x25, y25 = m.pos(c25)
    c26 = m.add(x25 + C30, y25 + 0.5); m.bond(c25, c26)
    c27 = m.add(x25, y25 - 1.0); m.bond(c25, c27)
    if extra_ethyl:
        x24, y24 = m.pos(ids[3])
        ex, ey = _outer(m, ids[3], [ids[2], ids[4]], 1.0)
        e1 = m.add(ex, ey); m.bond(ids[3], e1, hl=COL["nonpolar"])
        e2 = m.add(ex + C30, ey + (0.5 if ey > y24 else -0.5)); m.bond(e1, e2, hl=COL["nonpolar"])
    return ids


def cholesterol(name="Cholesterol", sitosterol=False):
    m = Mol(name)
    st = steroid_core(m, dbl=((5, 6),))
    ho = subst(m, st[3], -C30, -0.5, "HO", style="wedge", hl=COL["oh"])
    subst(m, st[10], 0.0, 0.95, "", style="wedge")     # C19 methyl
    subst(m, st[13], 0.0, 0.95, "", style="wedge")     # C18 methyl
    side_chain_cholestane(m, st, extra_ethyl=sitosterol)
    m.note(m.pos(st[3])[0] - 0.2, m.pos(st[3])[1] + 0.45, "3β", size=7.5)
    for k, lab in ((1, "A"), (2, "B"), (3, "C")):
        pass
    r3 = math.sqrt(3)
    for (cx, cy, lab) in ((0, 0, "A"), (r3, 0, "B"), (1.5 * r3, 1.5, "C")):
        m.note(cx, cy, lab, color="#9aa3b2", size=9, weight="bold")
    x13, y13 = m.pos(st[13]); x14, y14 = m.pos(st[14])
    m.note((x13 + x14) / 2 + 0.68, (y13 + y14) / 2, "D", color="#9aa3b2", size=9, weight="bold")
    m.st = st
    return m


def bile_acid(name="Glycocholate", oh=(("3", "a"), ("7", "a"), ("12", "a")), conj="glycine", ionized=True):
    """5-beta bile acid; oh entries (position, 'a' alpha=hash | 'b' beta=wedge)."""
    m = Mol(name)
    st = steroid_core(m)
    subst(m, st[10], 0.0, 0.95, "", style="wedge", hl=COL["nonpolar"])
    subst(m, st[13], 0.0, 0.95, "", style="wedge", hl=COL["nonpolar"])
    subst(m, st[5], 0.0, -0.85, "H", style="wedge")      # 5-beta H (A/B cis)
    vec = {"3": (-C30, -0.5), "7": (C30 * 0.0, -1.0), "12": (0.0, 1.0)}
    for pos, face in oh:
        dx, dy = vec[pos]
        if pos == "12":
            dx, dy = (-0.35, 0.9)
        if pos == "7":
            dx, dy = (0.35, -0.95)
        subst(m, st[int(pos)], dx, dy, "HO" if pos == "3" else "OH",
              style="hash" if face == "a" else "wedge", hl=COL["polar"])
    x, y = m.pos(st[17])
    c20 = m.add(x + 0.30, y + 0.95); m.bond(st[17], c20, 1, style="wedge")
    c21 = m.add(x - 0.45, y + 1.6); m.bond(c20, c21)
    ids = chain(m, c20, 3, heading=0, first=-1)
    c24 = ids[-1]
    x24, y24 = m.pos(c24)
    o24 = m.add(x24, y24 + 1.0, "O"); m.bond(c24, o24, 2)
    if conj == "glycine":
        nh = m.add(x24 + C30, y24 - 0.5, "NH"); m.bond(c24, nh, 1, hl=COL["amide"])
        ca = m.add(x24 + 2 * C30, y24); m.bond(nh, ca)
        cc = m.add(x24 + 3 * C30, y24 - 0.5); m.bond(ca, cc)
        m.bond(cc, m.add(x24 + 3 * C30, y24 - 1.5, "O"), 2)
        m.bond(cc, m.add(x24 + 4 * C30, y24, "O⁻" if ionized else "OH"), 1, hl=COL["polar"])
    else:
        m.bond(c24, m.add(x24 + C30, y24 - 0.5, "OH"), 1, hl=COL["polar"])
    m.st = st
    return m


# ---------------------------------------------------------------------------
# General 2D sketch helpers: rings, fused rings, substituents, reaction schemes
# (built on draw.py; every coordinate is explicit so orientation is fixed)
# ---------------------------------------------------------------------------
def _nbrs(m, i):
    out = []
    for b in m.bonds:
        if b["i"] == i:
            out.append(b["j"])
        elif b["j"] == i:
            out.append(b["i"])
    return out


def bond_between(m, i, j):
    for b in m.bonds:
        if (b["i"], b["j"]) in ((i, j), (j, i)):
            return b
    return None


def ring(m, cx, cy, n=6, rot=90.0, labels=None, side=1.0):
    """Regular n-ring with bond length `side`, first vertex at angle `rot` (deg),
    vertices counter-clockwise. labels: list of n labels ('' = carbon).
    Returns atom indices in ring order."""
    R = side / (2 * math.sin(math.pi / n))
    ids = []
    for k in range(n):
        a = math.radians(rot + 360.0 * k / n)
        ids.append(m.add(cx + R * math.cos(a), cy + R * math.sin(a), (labels[k] if labels else "")))
    for k in range(n):
        m.bond(ids[k], ids[(k + 1) % n], 1)
    m.rings = getattr(m, "rings", []) + [ids]
    return ids


def fuse(m, a, b, n=6, labels=None, side=1.0):
    """New regular n-ring sharing edge a-b, built on the side away from the
    existing neighbours. labels: for the n-2 NEW atoms, in order from b onward.
    Returns ring atoms [a, b, new...]."""
    (xa, ya), (xb, yb) = m.pos(a), m.pos(b)
    mx, my = (xa + xb) / 2, (ya + yb) / 2
    ex, ey = xb - xa, yb - ya
    L = math.hypot(ex, ey)
    px, py = -ey / L, ex / L
    # decide which side is free: away from centroid of current neighbours
    nb = [k for k in _nbrs(m, a) + _nbrs(m, b) if k not in (a, b)]
    if nb:
        cx_ = sum(m.atoms[k]["x"] for k in nb) / len(nb); cy_ = sum(m.atoms[k]["y"] for k in nb) / len(nb)
        if (cx_ - mx) * px + (cy_ - my) * py > 0:
            px, py = -px, -py
    apo = side / (2 * math.tan(math.pi / n))
    cx, cy = mx + px * apo, my + py * apo
    R = side / (2 * math.sin(math.pi / n))
    # walk from b around the new ring (away from a)
    ang_b = math.atan2(yb - cy, xb - cx)
    ang_a = math.atan2(ya - cy, xa - cx)
    step = 2 * math.pi / n
    # direction: going from b, the next vertex must not be a
    d = (ang_b - ang_a + math.pi) % (2 * math.pi) - math.pi
    sgn = 1 if d > 0 else -1
    ids = [a, b]
    prev = b
    for k in range(1, n - 1):
        ang = ang_b + sgn * step * k
        lab = labels[k - 1] if labels else ""
        i = m.add(cx + R * math.cos(ang), cy + R * math.sin(ang), lab)
        m.bond(prev, i, 1)
        ids.append(i); prev = i
    m.bond(prev, a, 1)
    m.rings = getattr(m, "rings", []) + [ids]
    return ids


def set_double(m, pairs):
    """pairs: iterable of (i, j) atom-index pairs to make double bonds."""
    for i, j in pairs:
        b = bond_between(m, i, j)
        if b is None:
            raise ValueError("no bond %d-%d" % (i, j))
        b["order"] = 2


def kekule(m, ids, start=0):
    """Alternate double bonds around ring `ids` beginning with bond start->start+1."""
    n = len(ids)
    for k in range(start, start + n, 2):
        bond_between(m, ids[k % n], ids[(k + 1) % n])["order"] = 2


def outward(m, i, dist=1.0):
    """Unit outward direction from atom i (away from its neighbours)."""
    x, y = m.pos(i)
    nb = _nbrs(m, i)
    sx = sum(m.atoms[k]["x"] - x for k in nb); sy = sum(m.atoms[k]["y"] - y for k in nb)
    L = math.hypot(sx, sy)
    if L < 1e-9:
        return (dist, 0.0)
    return (-sx / L * dist, -sy / L * dist)


def sub(m, i, label="", angle=None, dist=1.0, order=1, style="plain", hl=None, color=None, anchor=None):
    """Attach a substituent to atom i: outward by default, or at an absolute angle (deg)."""
    x, y = m.pos(i)
    if angle is None:
        dx, dy = outward(m, i, dist)
    else:
        dx, dy = dist * math.cos(math.radians(angle)), dist * math.sin(math.radians(angle))
    k = m.add(x + dx, y + dy, label, color=color, anchor=anchor)
    m.bond(i, k, order, style=style, hl=hl)
    return k


def grow(m, i, n, angle=None, first=+1, cis=(), trans=(), flip=()):
    """Zigzag chain of n atoms from atom i; heading = outward unless angle given."""
    if angle is None:
        dx, dy = outward(m, i)
        angle = math.degrees(math.atan2(dy, dx))
    return chain(m, i, n, heading=angle, first=first, cis=cis, trans=trans, flip=flip)


def label(m, i, text, anchor=None):
    m.atoms[i]["label"] = text
    m.atoms[i]["anchor"] = anchor


def hl_bonds(m, pairs, color):
    for i, j in pairs:
        bond_between(m, i, j)["hl"] = color


def hl_atoms_note(m, i, text, color=None, dist=0.75, size=8):
    x, y = m.pos(i)
    dx, dy = outward(m, i, dist)
    m.note(x + dx, y + dy, text, color=color or COL["muted"], size=size)


def shift(m, dx, dy):
    for a in m.atoms:
        a["x"] += dx; a["y"] += dy
    for n in m.notes:
        n["x"] += dx; n["y"] += dy
    return m


def rotate(m, deg, about=None):
    """Rotate a whole drawing (use to fix an orientation, e.g. to put a ring flat)."""
    if about is None:
        xs = [a["x"] for a in m.atoms]; ys = [a["y"] for a in m.atoms]
        about = ((min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2)
    c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
    for obj in m.atoms + m.notes:
        x, y = obj["x"] - about[0], obj["y"] - about[1]
        obj["x"], obj["y"] = about[0] + c * x - s * y, about[1] + s * x + c * y
    return m


def mirror(m, axis="x"):
    """Mirror left-right (axis='x') or top-bottom ('y'); wedges keep their sense."""
    for obj in m.atoms + m.notes:
        if axis == "x":
            obj["x"] = -obj["x"]
        else:
            obj["y"] = -obj["y"]
    return m


def merge(dst, src, dx=0.0, dy=0.0):
    """Copy all atoms/bonds/notes of src into dst with an offset; returns index map."""
    base = len(dst.atoms)
    for a in src.atoms:
        b = dict(a); b["x"] += dx; b["y"] += dy
        dst.atoms.append(b)
    for b in src.bonds:
        c = dict(b); c["i"] += base; c["j"] += base
        dst.bonds.append(c)
    for n in src.notes:
        c = dict(n); c["x"] += dx; c["y"] += dy
        dst.notes.append(c)
    return base


# ---------------------------------------------------------------------------
# Reaction / pathway scheme: molecules in a row (wrapping), arrows with labels
# ---------------------------------------------------------------------------
def scheme(items, title=None, per_row=3, unit=UNIT_IN, gap=3.2, caption=None, show_now=True):
    """items: [Mol, ("enzyme", "cofactor note"), Mol, (...), Mol ...]
    Molecules are drawn at one scale; arrows carry an enzyme label above and
    an optional cofactor/regulation note below. Rows wrap after per_row molecules."""
    mols = [x for x in items if isinstance(x, Mol)]
    arrows = [x for x in items if not isinstance(x, Mol)]
    exts = [extent(mm, pad=0.4) for mm in mols]
    rows = [list(range(k, min(k + per_row, len(mols)))) for k in range(0, len(mols), per_row)]
    row_h = [max(exts[k][3] - exts[k][2] for k in r) for r in rows]
    row_w = [sum(exts[k][1] - exts[k][0] for k in r) + gap * (len(r) - 1) for r in rows]
    W = max(row_w) + (1.6 if len(rows) > 1 else 0)
    H = sum(row_h) + 1.6 * (len(rows) - 1)
    th = 0.45 if title else 0.0
    ch = 0.4 if caption else 0.0
    fig = plt.figure(figsize=(W * unit, H * unit + th + ch))
    ax = fig.add_axes([0, ch / (H * unit + th + ch), 1, H * unit / (H * unit + th + ch)])
    ax.set_xlim(0, W); ax.set_ylim(-H, 0); ax.set_aspect("equal"); ax.axis("off")
    y_top = 0.0
    pos = {}
    for r, ids in enumerate(rows):
        x = 0.0 if r == 0 else 1.6
        cy = y_top - row_h[r] / 2
        for k in ids:
            x0, x1, y0, y1 = exts[k]
            dx = x - x0; dy = cy - (y0 + y1) / 2
            mm = Mol(); merge(mm, mols[k], dx, dy)
            draw(ax, mm)
            pos[k] = (x, x + (x1 - x0), cy, y1 - y0)
            x += (x1 - x0) + gap
        y_top -= row_h[r] + 1.6
    for k, lab in enumerate(arrows):
        if k + 1 >= len(mols):
            break
        enz, note = (lab, "") if isinstance(lab, str) else (lab[0], lab[1] if len(lab) > 1 else "")
        xa0, xa1, ya, ha = pos[k]; xb0, xb1, yb, hb = pos[k + 1]
        same = abs(ya - yb) < 1e-6
        if same:
            p, q = (xa1 + 0.25, ya), (xb0 - 0.25, yb)
        else:  # wrap: from under the last molecule of a row to above the first of the next
            p, q = ((xa0 + xa1) / 2, ya - ha / 2 + 0.1), ((xb0 + xb1) / 2, yb + hb / 2 - 0.1)
        ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=14, lw=1.4, color=COL["ink"],
                                     connectionstyle="arc3,rad=0" if same else "arc3,rad=-0.12"))
        mx, my = (p[0] + q[0]) / 2, (p[1] + q[1]) / 2
        box = dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9)
        if same:
            ax.text(mx, my + 0.3, enz, ha="center", va="bottom", fontsize=8.5, color=COL["ester"],
                    fontweight="bold", bbox=box)
            if note:
                ax.text(mx, my - 0.3, note, ha="center", va="top", fontsize=8, color=COL["muted"], bbox=box)
        else:
            ax.text(mx, my, enz + ("\n" + note if note else ""), ha="center", va="center", fontsize=8.5,
                    color=COL["ester"], bbox=box)
    if title:
        fig.text(0.0, 1 - (th / 2) / (H * unit + th + ch), title, fontsize=11.5, fontweight="bold", va="center")
    if caption:
        fig.text(0.0, (ch / 2) / (H * unit + th + ch), caption, fontsize=9, color=COL["muted"], va="center")
    if show_now:
        plt.show()
    return fig


try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    def Markdown(s): return s


def _md(lines):
    display(Markdown("\n".join(lines) if isinstance(lines, (list, tuple)) else lines))


# ---------------------------------------------------------------------------
# L29 lecture-notes figures and calculators: QSAR I (log 1/C, log P, pi, sigma,
# steric descriptors). Template: every substituent X sits at C4 (top) of the
# same benzoic acid ring, with the CO2H group at C1 (bottom), the system in
# which Hammett sigma is defined.
# ---------------------------------------------------------------------------
import math as _m
import numpy as _np

# Standard tabulated aromatic substituent constants (Hansch, Leo and Taft
# compilation, Chem. Rev. 1991): sigma_p, sigma_m, pi (from benzene, log P 2.13)
# and molar refractivity MR (cm3/mol).
QSUB = [  # (X, label for drawing, sigma_p, sigma_m, pi, MR)
    ("H",    "H",    0.00,  0.00,  0.00, 1.03),
    ("CH3",  "CH₃", -0.17, -0.07,  0.56, 5.65),
    ("Cl",   "Cl",   0.23,  0.37,  0.71, 6.03),
    ("OCH3", "OCH₃", -0.27, 0.12, -0.02, 7.87),
    ("NO2",  "NO₂",  0.78,  0.71, -0.28, 7.36),
    ("CF3",  "CF₃",  0.54,  0.43,  0.88, 5.02),
    ("OH",   "OH",  -0.37,  0.12, -0.67, 2.85),
    ("NH2",  "NH₂", -0.66, -0.16, -1.23, 5.42),
]
QSUB_FORMULA = {"H": "C7H6O2", "CH3": "C8H8O2", "Cl": "C7H5ClO2", "OCH3": "C8H8O3",
                "NO2": "C7H5NO4", "CF3": "C8H5F3O2", "OH": "C7H6O3", "NH2": "C7H7NO2"}


def x_benzoic(x="H"):
    """4-X-benzoic acid in fixed orientation: X up (C4), CO2H down (C1)."""
    lab = dict((r[0], r[1]) for r in QSUB)[x]
    m = Mol("4-%s-benzoic acid" % x)
    r = ring(m, 0, 0, 6, rot=90)
    kekule(m, r, 0)
    xc = COL["drug"] if x != "H" else COL["muted"]
    sub(m, r[0], lab, angle=90, color=xc)
    c = sub(m, r[3], "", angle=270)
    cx, cy = m.pos(c)
    m.bond(c, m.add(cx - C30, cy - 0.5, "O"), 2)
    m.bond(c, m.add(cx + C30, cy - 0.5, "OH"), 1, hl=COL["oh"])
    return m


def fig_substituent_set():
    mols = [x_benzoic(r[0]) for r in QSUB]
    titles = ["X = %s   σp %+.2f   π %+.2f" % (r[1], r[2], r[4]) for r in QSUB]
    grid(mols, titles, ncols=4, align="center")
    rows = ["| X | σp | σm | π | MR (cm³/mol) | Electronic effect at para | Hydrophobicity vs H |",
            "|---|---|---|---|---|---|---|"]
    for x, lab, sp, sm, pi, mr in QSUB:
        el = "reference" if x == "H" else ("withdrawing" if sp > 0 else "donating")
        hy = "reference" if x == "H" else ("more hydrophobic" if pi > 0 else "less hydrophobic")
        rows.append("| %s | %+.2f | %+.2f | %+.2f | %.2f | %s | %s |" % (lab, sp, sm, pi, mr, el, hy))
    _md(["Every analogue is drawn on one template: **X at C4 (top, red)** and the ionizable CO₂H at C1 (bottom), "
         "the reference reaction used to define σ. Values below are standard tabulated constants "
         "(Hansch–Leo–Taft compilation); π is measured relative to benzene (log P 2.13).", ""] + rows +
        ["", "Note how OCH₃ and OH are **σm > 0 but σp < 0**: at meta only their inductive pull is felt; at para their lone-pair "
         "resonance donation wins."])


def fig_logP_shapes(k2=0.25, k1=1.20, c=1.50):
    """A: straight line over a narrow log P range; B: parabola from an illustrative equation."""
    fig, ax = plt.subplots(1, 2, figsize=(10.2, 3.9))
    xs = _np.linspace(0.5, 2.5, 50)
    ax[0].plot(xs, 0.6 * xs + 2.2, color=COL["ester"], lw=2)
    xp = _np.array([0.6, 0.9, 1.3, 1.6, 1.9, 2.3])
    yp = 0.6 * xp + 2.2 + _np.array([0.05, -0.07, 0.04, -0.03, 0.06, -0.04])
    ax[0].plot(xp, yp, "o", color=COL["ink"], ms=5)
    ax[0].set_title("A. Narrow log P range: looks linear\nlog(1/C) = 0.60 log P + 2.20 (illustrative)", fontsize=10)
    ax[0].set_xlabel("log P"); ax[0].set_ylabel("log(1/C)  (higher = more potent)")
    p0 = k1 / (2 * k2)
    ymax = -k2 * p0 ** 2 + k1 * p0 + c
    xs = _np.linspace(-1, 5.8, 200)
    ax[1].plot(xs, -k2 * xs ** 2 + k1 * xs + c, color=COL["ester"], lw=2)
    ax[1].axvline(p0, color=COL["drug"], ls="--", lw=1.2)
    ax[1].plot([p0], [ymax], "o", color=COL["drug"], ms=7)
    ax[1].annotate("optimum  log P₀ = k₁/(2k₂)\n= %.2f/(2×%.2f) = %.2f" % (k1, k2, p0), (p0, ymax),
                   xytext=(p0 + 0.3, ymax - 1.6), fontsize=9, color=COL["drug"])
    ax[1].text(-0.9, ymax + 0.45, "too polar:\npoor membrane\ncrossing", fontsize=8.5, color=COL["muted"], va="top")
    ax[1].text(5.7, ymax + 0.45, "too lipophilic:\ntrapped in fat,\nprotein bound,\nmetabolized", fontsize=8.5, color=COL["muted"], va="top", ha="right")
    ax[1].set_title("B. Wide log P range: parabola\nlog(1/C) = −%.2f(log P)² + %.2f log P + %.2f (illustrative)" % (k2, k1, c), fontsize=10)
    ax[1].set_xlabel("log P")
    ax[1].set_ylim(ymax - 3.6, ymax + 0.6)
    for a in ax:
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def fig_hammett_rho():
    """log(K_X/K_H) = rho * sigma for three reference reactions (lines from the Hammett equation)."""
    rhos = [("benzoic acids (ρ = 1.00, the definition)", 1.00, COL["ink"]),
            ("phenols (ρ ≈ 2.2)", 2.23, COL["drug"]),
            ("phenylacetic acids (ρ ≈ 0.49)", 0.49, COL["ester"])]
    fig, ax = plt.subplots(figsize=(6.6, 4.3))
    xs = _np.linspace(-0.75, 0.85, 20)
    for lab, rho, col in rhos:
        ax.plot(xs, rho * xs, color=col, lw=2, label=lab)
    for x, lab, sp, sm, pi, mr in QSUB:
        ax.plot([sp], [sp], "o", color=COL["ink"], ms=4)
        ax.annotate(lab, (sp, sp), xytext=(4, -11), textcoords="offset points", fontsize=8)
    ax.axhline(0, color=COL["muted"], lw=0.8); ax.axvline(0, color=COL["muted"], lw=0.8)
    ax.set_xlabel("σp of X"); ax.set_ylabel("log(K_X / K_H)")
    ax.set_title("Hammett equation: log(K_X/K_H) = ρσ\nρ measures how sensitive a reaction is to substituent electronics", fontsize=10)
    ax.legend(fontsize=8.5, loc="upper left"); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


# ---- calculators ----------------------------------------------------------
def log_inv_C(conc_molar):
    _md(["1. C = **%.3g M**" % conc_molar,
         "2. 1/C = 1 / %.3g = **%.3g M⁻¹**" % (conc_molar, 1 / conc_molar),
         "3. log(1/C) = log₁₀(%.3g) = **%.2f**" % (1 / conc_molar, _m.log10(1 / conc_molar)),
         "",
         "A ten-fold lower C raises log(1/C) by exactly 1 unit, so the scale rises with potency."])
    return dict(log_inv_C=_m.log10(1 / conc_molar))


def pi_from_logP(logP_parent=2.13, logP_X=2.84, name_X="chlorobenzene"):
    pi = logP_X - logP_parent
    _md(["1. π_X = log P(Ar–X) − log P(Ar–H)",
         "2. π_X = %.2f − %.2f = **%+.2f** (%s)" % (logP_X, logP_parent, pi, name_X),
         "3. Interpretation: π %s 0, so X is **%s** than H at this position (P is %.2g× %s)."
         % (">" if pi > 0 else ("<" if pi < 0 else "="),
            "more hydrophobic" if pi > 0 else ("less hydrophobic" if pi < 0 else "no different"),
            10 ** abs(pi), "higher" if pi >= 0 else "lower")])
    return dict(pi=pi)


def logP_from_pi(logP_parent=2.13, pis=(0.71, -0.67)):
    tot = logP_parent + sum(pis)
    _md(["1. log P(new) = log P(parent) + Σπ  (assumes the substituent effects are additive)",
         "2. log P(new) = %.2f %s = **%.2f**" % (logP_parent, " ".join("%+.2f" % p for p in pis), tot)])
    return dict(logP=tot)


def sigma_from_pKa(pKa_H=4.20, pKa_X=3.44, name_X="4-nitrobenzoic acid"):
    s = pKa_H - pKa_X
    _md(["1. Hammett definition (benzoic acids, water, 25 °C, ρ = 1): σ_X = log K_X − log K_H",
         "2. Because pKa = −log K: σ_X = pKa(H) − pKa(X)",
         "3. σ_X = %.2f − %.2f = **%+.2f** (%s)" % (pKa_H, pKa_X, s, name_X),
         "4. Interpretation: σ %s 0, so X is electron-**%s**: it %s the benzoate anion and makes the acid %s (K_X/K_H = %.2f)."
         % (">" if s > 0 else "<", "withdrawing" if s > 0 else "donating",
            "stabilizes" if s > 0 else "destabilizes", "stronger" if s > 0 else "weaker", 10 ** s)])
    return dict(sigma=s, K_ratio=10 ** s)


def logP_optimum(k2=0.25, k1=1.20, c=1.50):
    p0 = k1 / (2 * k2)
    best = -k2 * p0 ** 2 + k1 * p0 + c
    lines = ["Equation: log(1/C) = −%.3f(log P)² + %.3f log P + %.3f" % (k2, k1, c),
             "1. At the top of the curve the slope is zero: d[log(1/C)]/d(log P) = −2k₂ log P + k₁ = 0",
             "2. log P₀ = k₁ / (2k₂) = %.3f / (2 × %.3f) = **%.2f**" % (k1, k2, p0),
             "3. Maximum log(1/C) = −%.3f(%.2f)² + %.3f(%.2f) + %.3f = **%.2f**" % (k2, p0, k1, p0, c, best),
             "", "| log P | log(1/C) | vs optimum |", "|---|---|---|"]
    for lp in (p0 - 2, p0 - 1, p0, p0 + 1, p0 + 2):
        y = -k2 * lp ** 2 + k1 * lp + c
        lines.append("| %.2f | %.2f | %+.2f |" % (lp, y, y - best))
    lines += ["", "Equal steps either side of log P₀ lose equal activity, because the parabola is symmetric."]
    _md(lines)
    return dict(logP0=p0, max_activity=best)


# Illustrative data set (made up for teaching; NOT measured data): eight
# 4-X analogues with log(1/C) built from 1.0*pi + 0.9*sigma_p + 4.0 plus small
# deviations standing in for assay error.
QDATA = {"H": 4.05, "CH3": 4.33, "Cl": 5.02, "OCH3": 3.80, "NO2": 4.30, "CF3": 5.41, "OH": 2.95, "NH2": 2.25}


def _ols(X, y):
    A = _np.column_stack([X, _np.ones(len(y))])
    beta = _np.linalg.lstsq(A, y, rcond=None)[0]
    return beta, A @ beta


def qsar_fit(use_pi=True, use_sigma=False, use_MR=False, show_plot=True):
    names = [n for n, on in (("π", use_pi), ("σp", use_sigma), ("MR/10", use_MR)) if on]
    if not names:
        _md(["Switch on at least one descriptor."]); return {}
    cols = {"π": 4, "σp": 2, "MR/10": 5}
    subs = [r[0] for r in QSUB]
    X = _np.array([[r[cols[n]] / (10.0 if n == "MR/10" else 1.0) for n in names] for r in QSUB])
    y = _np.array([QDATA[s] for s in subs])
    n, k = len(y), len(names)
    beta, pred = _ols(X, y)
    ss_tot = float(((y - y.mean()) ** 2).sum()); ss_res = float(((y - pred) ** 2).sum())
    r2 = 1 - ss_res / ss_tot
    s = _m.sqrt(ss_res / (n - k - 1))
    F = (r2 / k) / ((1 - r2) / (n - k - 1))
    press = 0.0; loo = []
    for i in range(n):
        keep = [j for j in range(n) if j != i]
        b_i, _ = _ols(X[keep], y[keep])
        yp = float(_np.dot(X[i], b_i[:-1]) + b_i[-1]); loo.append(yp)
        press += (y[i] - yp) ** 2
    q2 = 1 - press / ss_tot
    eq = "log(1/C) = " + " ".join("%+.3f %s" % (b, nm) for b, nm in zip(beta[:-1], names)) + " %+.3f" % beta[-1]
    rows = ["*Illustrative data set (invented for teaching, not measured).*", "",
            "| X | " + " | ".join(names) + " | log(1/C) obs | fitted | left-out prediction |",
            "|---|" + "---|" * (len(names) + 3)]
    for i, sb in enumerate(subs):
        rows.append("| %s | " % QSUB[i][1] + " | ".join("%+.2f" % v for v in X[i]) +
                    " | %.2f | %.2f | %.2f |" % (y[i], pred[i], loo[i]))
    rows += ["", "1. Least-squares fit: **%s**" % eq,
             "2. n = %d compounds, k = %d descriptor(s); compounds per descriptor = %.1f %s"
             % (n, k, n / k, "(meets the ≥ 5 rule)" if n / k >= 5 else "(**below** the ≥ 5 rule: treat with caution)"),
             "3. SS_total = Σ(y − ȳ)² = %.3f;  SS_residual = Σ(y − ŷ)² = %.3f" % (ss_tot, ss_res),
             "4. r² = 1 − SS_res/SS_tot = 1 − %.3f/%.3f = **%.3f**  (r = %.3f; %.1f%% of the variance explained)" % (ss_res, ss_tot, r2, _m.sqrt(max(r2, 0)), 100 * r2),
             "5. s = √[SS_res/(n − k − 1)] = √(%.3f/%d) = **%.3f** log units" % (ss_res, n - k - 1, s),
             "6. F = (r²/k) / [(1 − r²)/(n − k − 1)] = (%.3f/%d) / (%.3f/%d) = **%.1f**" % (r2, k, 1 - r2, n - k - 1, F),
             "7. Leave-one-out: refit without each compound and predict it; PRESS = Σ(y − ŷ_loo)² = %.3f" % press,
             "8. q² = 1 − PRESS/SS_tot = 1 − %.3f/%.3f = **%.3f**" % (press, ss_tot, q2)]
    _md(rows)
    if show_plot:
        fig, ax = plt.subplots(figsize=(4.8, 4.4))
        ax.plot(pred, y, "o", color=COL["ester"], label="fitted")
        ax.plot(loo, y, "s", mfc="none", color=COL["dbl"], label="left-out prediction")
        for i in range(n):
            ax.annotate(QSUB[i][1], (pred[i], y[i]), xytext=(4, -13) if QSUB[i][0] == "NO2" else ((5, -11) if QSUB[i][0] == "CF3" else (4, 3)), textcoords="offset points", fontsize=8)
        lo, hi = min(y.min(), min(loo)) - 0.3, max(y.max(), max(loo)) + 0.3
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_xlabel("predicted log(1/C)"); ax.set_ylabel("observed log(1/C)")
        ax.set_title("%s\nr² = %.3f   q² = %.3f   s = %.3f" % (" + ".join(names), r2, q2, s), fontsize=10)
        ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()
    return dict(beta=list(beta), r2=r2, s=s, F=F, q2=q2, n=n, k=k)

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110})
# ===== END kit =====


## Learning objectives

| # | By the end of this activity you should be able to | Bloom | Syllabus level | SILO |
|---|---|---|---|---|
| 1 | Interpret a linear versus a parabolic relationship between activity and a descriptor. | Understand | V | 3 |
| 2 | Interpret the sign and size of a substituent constant and predict its effect on activity. | Analyze | III | 3 |
| 3 | Distinguish hydrophobic, electronic and steric descriptors and state when each applies. | Understand | V | 3 |
| 4 | Judge whether a correlation coefficient justifies the claim made from it. | Evaluate | II | 6 |

*Syllabus levels follow the PHS 312 legend, in which Level I is Creating and Level VI is Remembering.*

## Part 1 - What do you already know?

Three questions before you begin. **You will not be told whether you are
right**, and that is deliberate: the point is to measure what you knew coming
in, so that at the end you can see how much the activity moved you. Answer
from what you remember from the reading. Guessing is fine.

In [ ]:
#@title P1   (run after choosing) { display-mode: "form" }
#@markdown **P1.**  Biological activity is entered into a QSAR equation as log(1/C). Why the reciprocal?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. So that negative activity values can never be generated", "B. So that the units of concentration cancel in the equation", "C. So that the scale rises as the concentration needed falls", "D. So that the equation becomes linear in every descriptor"]
gb.answer("P1", response[:1])

In [ ]:
#@title P2   (run after choosing) { display-mode: "form" }
#@markdown **P2.**  A substituent has pi = 1.68. What does that positive value indicate?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It is more hydrophobic than a hydrogen at the same position", "B. It is more electron-withdrawing than a hydrogen at that site", "C. It is larger in van der Waals volume than a hydrogen atom", "D. It is more readily metabolized than a hydrogen at that site"]
gb.answer("P2", response[:1])

In [ ]:
#@title P3   (run after choosing) { display-mode: "form" }
#@markdown **P3.**  An equation contains a negative coefficient on a squared log P term. What does the curve look like?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It rises steadily with log P across the whole range studied", "B. It is flat, since the squared term cancels the linear term", "C. It falls to a minimum at an optimum log P and then increases", "D. It rises to a maximum at an optimum log P and then declines"]
gb.answer("P3", response[:1])

## Part 2 - Key ideas: what QSAR measures and why activity is written as log(1/C)


**Summary.** A QSAR (quantitative structure–activity relationship) is an equation that predicts the potency of compounds in one structural series from a few numbers that describe them. Ordinary SAR says *"a chlorine here helps"*; QSAR says *by how much, and which property of the chlorine is responsible*. It needs two things: a reliable activity measurement for every compound, and descriptors measured or tabulated on a common scale.

**The activity scale.** Potency is usually measured as the concentration C (in mol/L) that gives a standard effect (an IC₅₀, EC₅₀, Kᵢ or minimum effective dose). Two problems make raw C awkward: a *more* potent compound has a *smaller* C, and potencies in one series can span a thousand-fold or more. Taking **log(1/C)** (the same as −log C, e.g. pIC₅₀) fixes both: the number goes **up** as potency goes up, and each unit is a ten-fold change. Free-energy terms such as binding energy are proportional to log K, so descriptors that act through binding are expected to add linearly on this log scale.

**The three descriptor families.** Most of what a substituent does to binding and transport can be sorted into three questions: *How greasy is it? How does it push or pull electrons? How big is it?*

| Family | Descriptor | Measured from | Positive value means | Typical role in a drug |
|---|---|---|---|---|
| Hydrophobic (whole molecule) | **log P** | octanol/water partition of the compound | prefers octanol (lipophilic) | absorption, membrane and blood–brain-barrier crossing, protein binding, metabolism |
| Hydrophobic (substituent) | **π** = log P(X) − log P(H) | log P of substituted vs parent compound | X more hydrophobic than H | contact with a hydrophobic pocket at that position |
| Electronic (aromatic) | **Hammett σ** (σm, σp) | ionization of substituted benzoic acids | X withdraws electrons | pKa of nearby groups, H-bond strength, charge on the pharmacophore |
| Electronic (parts of σ) | **F** (field/inductive), **R** (resonance) | statistical split of σm and σp | withdraws by that route | which electronic route matters at a position |
| Electronic (aliphatic) | **σ\*** / **σI** | ester hydrolysis rates (Taft) | withdraws inductively | groups on sp³ carbon, where resonance cannot operate |
| Steric | **Taft Eₛ** | acid-catalysed ester hydrolysis rate | *less* hindering than the reference (more negative = bulkier) | crowding near a reacting or binding centre |
| Steric / bulk | **MR** (molar refractivity) | refractive index, molar mass and density | larger, more polarizable group | filling a pocket; dispersion contacts |
| Steric (shape) | **Verloop STERIMOL** (L, B1…B5) | computed from bond lengths, angles and van der Waals radii | longer or wider in that direction | fit in a narrow or shaped cleft |

<details>
<summary><b>▸ Key terms (open)</b></summary>


| Term | Meaning |
|---|---|
| QSAR | An equation relating a measured activity of a series to physicochemical descriptors |
| Congeneric series | Compounds sharing one scaffold and differing only in substituents; a QSAR applies only inside one such series |
| log(1/C) | Activity scale: C is the molar concentration giving a fixed effect; higher = more potent |
| P, log P | Partition coefficient (octanol/water) of the neutral compound, and its log |
| π | Substituent hydrophobicity constant, log P of the substituted compound minus that of the parent |
| log P₀ | The log P at which activity is greatest in a parabolic relationship |
| σ (Hammett) | Substituent electronic constant from benzoic acid ionization; + = electron-withdrawing, − = electron-donating |
| ρ (Hammett) | Sensitivity of a reaction or equilibrium to σ; 1.00 for benzoic acid ionization by definition |
| F, R | Field (inductive) and resonance components of a substituent's electronic effect |
| σ\*, σI | Inductive constants for substituents on aliphatic (sp³) carbon |
| Eₛ | Taft steric constant from ester hydrolysis rates; more negative = bulkier |
| MR | Molar refractivity, a measure of substituent volume and polarizability |
| Verloop parameters | Computed length and width dimensions of a substituent (STERIMOL) |

</details>

#### Worked example: converting potency to log(1/C)
Compound A gives 50% inhibition at 2.0 µM and compound B at 20 nM. Express each as log(1/C) and say how many log units (and how many fold) B is more potent.

<details>
<summary><b>▸ Worked solution</b></summary>

1. Convert to mol/L: A = 2.0 × 10⁻⁶ M; B = 2.0 × 10⁻⁸ M.
2. log(1/C) for A = −log(2.0 × 10⁻⁶) = 6 − 0.30 = **5.70**.
3. log(1/C) for B = −log(2.0 × 10⁻⁸) = 8 − 0.30 = **7.70**.
4. Difference = 7.70 − 5.70 = **2.00 log units = 10² = 100-fold** more potent.

The more potent compound has the larger number, which is why log(1/C) rather than C goes on the y-axis. Check with cell 2.1.
</details>

In [ ]:
#@title 2.1 Calculator — log(1/C) from a molar concentration { display-mode: "form" }
concentration_molar = 2.0e-6 #@param {type:"number"}
_ = log_inv_C(concentration_molar)

## Part 3 - Hydrophobicity: log P, the substituent constant π and the log P optimum


**log P for the whole molecule.** P is the ratio of the concentration of the neutral compound in 1-octanol to that in water at equilibrium; octanol stands in for membrane lipid. A drug must be lipophilic enough to cross membranes but polar enough to dissolve in plasma and reach its target, so log P often appears in a QSAR even when the target pocket itself does not care about hydrophobicity.

* Over a **narrow** range of log P, activity often rises in a straight line (binding to serum albumin is a classic linear case).
* Over a **wide** range, activity usually rises, peaks and falls: very polar compounds cannot cross membranes; very lipophilic ones dissolve in fat depots, bind plasma proteins, are metabolized quickly, or are poorly soluble. The fit is then a **parabola**, log(1/C) = −k₂(log P)² + k₁ log P + k₃, and the best value, **log P₀ = k₁/(2k₂)**, is where the slope is zero.
* For general anaesthetics of several unrelated classes log P₀ is close to 2.3, and potent CNS-active hypnotics sit near log P 2. The flip side is a design rule: moving log P **away** from about 2 is one way to reduce unwanted CNS effects of a peripherally acting drug.

**π for one substituent.** π_X = log P(R–X) − log P(R–H). A positive π means X is more hydrophobic than H; negative means more hydrophilic. Because π values are roughly **additive**, log P of a new analogue can be estimated as log P(parent) + Σπ. Additivity breaks down when substituents interact (through-ring electronic effects, intramolecular H-bonds) or when the parent differs a lot from the one the π values were measured on (aromatic and aliphatic π values differ). A QSAR can contain both: **log P** for overall transport, **π** for a specific pocket that contacts one position.

In [ ]:
#@title 3.1 Figure — the eight-substituent set on one 4-X-benzoic acid template, with σ and π { display-mode: "form" }
fig_substituent_set()

In [ ]:
#@title 3.2 Calculator — π from two log P values { display-mode: "form" }
logP_parent = 2.13 #@param {type:"number"}
logP_substituted = 2.84 #@param {type:"number"}
name_of_compound = "chlorobenzene" #@param {type:"string"}
_ = pi_from_logP(logP_parent, logP_substituted, name_of_compound)

#### Worked example: π from log P, then log P from π
Benzene has log P 2.13 and phenol 1.46. (a) Calculate π for OH. (b) Using π(Cl) = +0.71, estimate log P of 4-chlorophenol. (c) The measured value is about 2.4. Why might the estimate be low?

<details>
<summary><b>▸ Worked solution</b></summary>

(a) π(OH) = 1.46 − 2.13 = **−0.67**: OH is about 4.7-fold (10^0.67) less hydrophobic than H.

(b) log P(4-chlorophenol) ≈ 2.13 + 0.71 + (−0.67) = **2.17**.

(c) π values assume each group acts independently. In 4-chlorophenol the electron-withdrawing Cl, para to OH, makes the O–H more acidic and changes how strongly the OH is hydrated, so the two groups are not independent. A gap of about 0.2 log unit is typical of this kind of electronic interaction. Check (a) with cell 3.2 and (b) with cell 3.3.
</details>

In [ ]:
#@title 3.3 Calculator — log P of a new analogue from parent log P + Σπ { display-mode: "form" }
logP_parent = 2.13 #@param {type:"number"}
pi_1 = 0.71 #@param {type:"number"}
pi_2 = -0.67 #@param {type:"number"}
_ = logP_from_pi(logP_parent, (pi_1, pi_2))

In [ ]:
#@title 3.4 Figure — straight-line and parabolic dependence of activity on log P (illustrative equations) { display-mode: "form" }
fig_logP_shapes()

In [ ]:
#@title 3.5 Calculator — optimum log P₀ from a parabolic equation (enter k₂ as a positive number) { display-mode: "form" }
k2_squared_term = 0.25 #@param {type:"number"}
k1_linear_term = 1.20 #@param {type:"number"}
k3_constant = 1.50 #@param {type:"number"}
_ = logP_optimum(k2_squared_term, k1_linear_term, k3_constant)

#### Worked example: finding log P₀
A series fits log(1/C) = −0.18(log P)² + 0.81 log P + 2.00 (illustrative). Find log P₀ and the predicted maximum log(1/C). An analogue has log P 4.0; by how much would bringing it to log P₀ raise log(1/C)?

<details>
<summary><b>▸ Worked solution</b></summary>

1. log P₀ = k₁/(2k₂) = 0.81/(2 × 0.18) = 0.81/0.36 = **2.25**.
2. Maximum = −0.18(2.25)² + 0.81(2.25) + 2.00 = −0.911 + 1.823 + 2.00 = **2.91**.
3. At log P 4.0: −0.18(16) + 0.81(4.0) + 2.00 = −2.88 + 3.24 + 2.00 = **2.36**.
4. Gain = 2.91 − 2.36 = **0.55 log unit ≈ 3.5-fold**.

Enter k₂ = 0.18, k₁ = 0.81, k₃ = 2.00 in cell 3.5 to check.
</details>

## Part 4 - Electronic and steric descriptors: σ, ρ, F and R, Eₛ, MR and Verloop


**Hammett σ.** σ is defined from the ionization of benzoic acids in water at 25 °C: σ_X = log K(X-benzoic acid) − log K(benzoic acid), which equals **pKa(H) − pKa(X)**.

* An **electron-withdrawing** X (NO₂, CF₃, Cl) spreads the negative charge of the carboxylate, so the anion is more stable, the acid is stronger and **σ is positive**.
* An **electron-donating** X (CH₃, NH₂, OCH₃ at para) concentrates charge on the carboxylate, the acid is weaker and **σ is negative**.
* σ is **position-dependent**: σm reflects mainly the inductive (through-bond, field) effect; σp adds resonance. Groups with a lone pair next to the ring (OH, OCH₃, NH₂) are inductively withdrawing but donate by resonance, so σm is small or positive while σp is clearly negative.
* **Ortho** values are not tabulated as ordinary σ: a group next to the carboxyl also crowds it and can H-bond to it, so its steric and proximity effects cannot be separated from its electronic effect.

**ρ (rho).** For any other reaction or equilibrium of a substituted benzene, log(K_X/K_H) = **ρσ**. ρ = 1 for benzoic acids by definition; a larger ρ means that process is more sensitive to substituent electronics (phenol ionization, where the charge sits closer to the ring, has ρ ≈ 2.2). In a drug QSAR, the fitted coefficient of σ plays the role of ρ for the biological response.

**F and R; aliphatic σ.** Hansch and colleagues split σ into a **field/inductive** part F and a **resonance** part R, so an equation can show which route matters. For substituents on sp³ carbon, where resonance cannot reach, Taft's inductive constants (**σ\***, **σI**) come from comparing base- and acid-catalysed ester hydrolysis rates.

In [ ]:
#@title 4.1 Figure — the Hammett equation for three reactions with different ρ { display-mode: "form" }
fig_hammett_rho()

In [ ]:
#@title 4.2 Calculator — Hammett σ from benzoic acid pKa values { display-mode: "form" }
pKa_benzoic_acid = 4.20 #@param {type:"number"}
pKa_substituted = 3.44 #@param {type:"number"}
name_of_acid = "4-nitrobenzoic acid" #@param {type:"string"}
_ = sigma_from_pKa(pKa_benzoic_acid, pKa_substituted, name_of_acid)

#### Worked example: σ from pKa, and why meta and para differ
Benzoic acid has pKa 4.20. 4-Chlorobenzoic acid has pKa 3.98 and 3-chlorobenzoic acid 3.83; 4-methoxybenzoic acid has pKa 4.47. Calculate σp(Cl), σm(Cl) and σp(OCH₃) and explain the signs.

<details>
<summary><b>▸ Worked solution</b></summary>

σ = pKa(H) − pKa(X).

* σp(Cl) = 4.20 − 3.98 = **+0.22**
* σm(Cl) = 4.20 − 3.83 = **+0.37**
* σp(OCH₃) = 4.20 − 4.47 = **−0.27**

Cl withdraws inductively at both positions, but at para it also donates a little by resonance through its lone pairs, which partly offsets the pull; so σp < σm. OCH₃ is inductively withdrawing, but its oxygen lone pair donates strongly by resonance to the para position, so the net σp is negative (the anion is destabilized and the acid is weaker). The tabulated values (+0.23, +0.37, −0.27) match to within measurement error. Check with cell 4.2.
</details>

**Steric descriptors** answer *"how big is it, and in which direction?"* No single number does this well, so three are in use:

| Descriptor | How obtained | Strength | Limitation |
|---|---|---|---|
| **Taft Eₛ** | log k(X) − log k(reference) for **acid-catalysed** hydrolysis of esters X–CO₂R, where rate depends mainly on crowding of the tetrahedral intermediate | experimental; directly a "crowding" measure | only for groups that act purely sterically on that intermediate (not by resonance or H-bonding); reference scale varies (CH₃ = 0 in Taft's original, H = 0 in many QSAR tables); may underestimate bulk in binding to a protein |
| **MR** (molar refractivity) | from refractive index, molar mass and density; additive over atoms | available for almost any group; encodes both volume and polarizability | a single number, so a long thin group and a short wide one can have the same MR; tracks π closely in homologous alkyl series |
| **Verloop STERIMOL** (L, B1…B5) | computed from standard bond lengths, angles and van der Waals radii | gives length and widths separately; any substituent | several numbers per group, so more descriptors to fit (watch the compound-per-descriptor rule) |

Because MR and π both rise steadily through H, CH₃, C₂H₅, n-C₃H₇, n-C₄H₉, a series built only from alkyl groups cannot tell a size effect from a hydrophobic one. Choosing substituents whose descriptors are **not correlated** is the subject of the Craig plot in L30.

#### Worked example: which steric descriptor?
Three situations: (a) a substituent on an sp³ carbon next to an ester that must be hydrolysed by an esterase; (b) a para substituent thought to fill a roomy hydrophobic pocket; (c) a substituent that must slot into a narrow cleft that tolerates length but not width. Which steric descriptor is the best first choice in each?

<details>
<summary><b>▸ Worked solution</b></summary>

(a) **Eₛ**: it was defined from exactly this kind of reaction, crowding of a tetrahedral intermediate at an ester carbon.

(b) **MR**: a roomy pocket rewards overall volume and polarizability (dispersion contacts); a single additive number is enough. Check whether MR is correlated with π in the chosen set before believing either term.

(c) **Verloop L and B parameters**: only these separate length from width, which is what a narrow cleft discriminates.
</details>

## Part 5 - One descriptor at a time

Eleven coumarin analogues inhibiting monoamine oxidase B. Each point is one
substituent, labelled.

Choose a descriptor and whether to fit a straight line or a parabola. Read
**r** (closer to 1 is a better fit) and **s** (the standard error — smaller
is better).

Begin with `pi` as a straight line. Then fit `pi` as a parabola.

<details>
<summary><b>Stuck? Open this before you ask a neighbour</b></summary>

Work through the descriptors in this order and the chapter's argument builds
itself.

1. `pi`, linear. A weak correlation — roughly r = 0.48. It would be easy to
   conclude hydrophobicity does not matter here.
2. `pi`, quadratic. The fit jumps to about r = 0.93. Hydrophobicity matters
   enormously; the *relationship* was simply not a straight line.
3. `Vdw` and `F`, either way. Both stay poor. Not every descriptor you can
   tabulate is relevant to a given series.

The lesson in step 2 is the one worth carrying: a poor linear fit is evidence
about the *shape* of the relationship, not proof that the property is
irrelevant.

</details>

In [ ]:
#@title Fit one descriptor { display-mode: "form" }
#@markdown Choose a descriptor and the form of the fit.
descriptor = "pi" #@param ["pi", "Vdw", "F"]
fit_a_parabola = False #@param {type:"boolean"}
r = sims.qsar_single(descriptor, fit_a_parabola)
print(f"descriptor        {r['param']}")
print(f"form of fit       {'quadratic' if r['quadratic'] else 'linear'}")
print(f"correlation r     {r['r']:.3f}")
print(f"standard error s  {r['s']:.3f}")

**Now put two settings side by side.** Run the cell below and compare the two figures it prints, one above the other.

In [ ]:
#@title Side by side: the same descriptor, straight line and parabola { display-mode: "form" }
#@markdown One descriptor, two functional forms.
which_descriptor = "pi" #@param ["pi", "Vdw", "F"]
out = []
for label, quad in (("STRAIGHT LINE", False), ("PARABOLA", True)):
    print("=" * 70); print(f"{label} - descriptor {which_descriptor}"); print("=" * 70)
    out.append((label, sims.qsar_single(which_descriptor, quad)))
print()
print(f"{'':16}{'r':>8}{'s':>8}")
for label, r in out:
    print(f"{label:16}{r['r']:>8.3f}{r['s']:>8.3f}")
lin, par = out[0][1]['r'], out[1][1]['r']
print(f"\nFitting a curve changed r from {lin:.3f} to {par:.3f}.")
print("Same data. Same descriptor. The relationship was never a straight line."
      if par - lin > 0.2 else
      "Little gained here - for this descriptor the curve is not the issue.")

In [ ]:
#@title C1   (run after choosing) { display-mode: "form" }
#@markdown **C1.**  Fitting pi as a straight line gave r = 0.48; as a parabola it gave r = 0.93. What does that difference establish?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Hydrophobicity matters, but the relationship is not a straight line", "B. Hydrophobicity does not matter, since the linear fit was so poor", "C. The parabola has over-fitted, since adding a term always helps r", "D. The activity data for these eleven analogues must be unreliable"]
gb.answer("C1", response[:1])

In [ ]:
#@title C2   (run after choosing) { display-mode: "form" }
#@markdown **C2.**  The van der Waals volume descriptor gave r near 0.01 linear and 0.12 quadratic. What is the sound conclusion?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Steric bulk does not govern activity across this particular series", "B. Steric bulk governs activity but needs a cubic term to be seen", "C. The volume values must have been tabulated for the wrong atoms", "D. Steric bulk matters only once hydrophobicity has been removed"]
gb.answer("C2", response[:1])

## Part 6 - Reading the statistics of a fitted equation (practice)


A QSAR is only as good as its statistics. Report every equation with:

| Statistic | What it tells you | Good sign |
|---|---|---|
| **n** | number of compounds | at least **5 per descriptor** (k) |
| **r** and **r²** | how closely fitted values follow observed ones; r² = fraction of the variance explained | r > 0.9 (r² > 0.8) for in-vitro data |
| **s** | standard error of the estimate, in log units | close to the assay's own error (often 0.2–0.3) |
| **F** | whether the equation beats a flat line through the mean, given n and k | well above the tabulated F for (k, n − k − 1) |
| **q²** | leave-one-out predictive r²: each compound predicted by an equation that never saw it | > 0.5, and not far below r² |

The cell below fits the **illustrative** eight-compound data set built on the substituent table in Part 3. Start with π alone, then add σp, then MR. Watch r² creep up while s, F and q² tell you whether the new term earned its place. L30 develops this further.

In [ ]:
#@title 6.1 Calculator — least-squares fit of log(1/C) with r², s, F and leave-one-out q² (illustrative data) { display-mode: "form" }
use_pi = True #@param {type:"boolean"}
use_sigma = False #@param {type:"boolean"}
use_MR = False #@param {type:"boolean"}
_ = qsar_fit(use_pi, use_sigma, use_MR)

#### Worked example: does adding a descriptor help?
With the illustrative data set in cell 6.1: π alone gives r² 0.899, s 0.354, q² 0.844. π + MR gives r² 0.902, s 0.381, q² 0.791. Should MR stay in the equation?

<details>
<summary><b>▸ Worked solution</b></summary>

1. r² rose by only 0.003. Adding any term, even a random one, can only raise r², so a tiny rise is not evidence.
2. s **rose** (0.354 → 0.381): the small fall in residual error was outweighed by losing a degree of freedom.
3. q² **fell** (0.844 → 0.791): the extra term makes predictions for unseen compounds worse.
4. n/k fell from 8 to 4, below the 5-per-descriptor rule.

**No**: drop MR. Now try π + σp in cell 6.1, where s falls sharply and q² rises, which is what a real effect looks like (though n/k = 4 still asks for more compounds before trusting it).
</details>

## Part 7 - QSAR thinking in real drug series (reference)


| Series / drug class | Descriptor that mattered | What the relationship showed | Mechanism-level take-home |
|---|---|---|---|
| **General anaesthetic ethers and other volatile anaesthetics** | log P (parabolic) | activity peaks near log P ≈ 2.3 across several structural classes | CNS penetration needs balanced lipophilicity, not maximal lipophilicity |
| **Barbiturates** (hypnotics) | log P | the most potent CNS-active members cluster near log P ≈ 2 | brain entry by passive diffusion across the blood–brain barrier |
| **β-blockers: propranolol vs atenolol** | log P | propranolol (log P ≈ 3) enters the CNS readily; atenolol (log P ≈ 0.2) much less | lowering log P away from ~2 reduces CNS side effects of a peripherally acting drug |
| **Drug binding to serum albumin** | log P (linear over the range studied) | binding rises steadily with lipophilicity | hydrophobic drugs are more highly protein-bound, lowering the free fraction |
| **Antibacterial sulfonamides** | pKa of the sulfonamide N–H (set by electronic effects of the ring substituent) | activity is greatest at intermediate pKa (about 6–7.4) | partly ionized drug both crosses the bacterial membrane (neutral) and mimics the p-aminobenzoate anion at dihydropteroate synthase (ionized) |
| **Diethyl phenyl phosphate insecticides** | σ of the ring substituent | electron-withdrawing groups increase activity | an electron-poor phenoxide is a better leaving group, so the phosphate phosphorylates the active-site serine of acetylcholinesterase faster |

**Looking ahead (L30 QSAR II).** L30 combines these descriptors in one **Hansch equation**, uses the **Craig plot** and **Topliss tree** to choose substituents, adds Free–Wilson indicator variables and moves to 3D-QSAR (CoMFA).

## Part 8 - End-of-activity assessment

Five questions in the style of the course quizzes and exams, mapped to the
Student Intended Learning Outcomes for this unit (**SILO 3**).
This time you get full feedback, including why each wrong option is wrong.
Answer each one, then run the summary cell at the bottom.

In [ ]:
#@title Q1   (run after choosing) { display-mode: "form" }
#@markdown **Q1.**  A substituent is reported with pi = 1.68 and sigma = -0.20. What does this pair describe?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Less hydrophobic than hydrogen and strongly electron-accepting", "B. More hydrophobic than hydrogen and weakly electron-donating", "C. More hydrophobic than hydrogen and strongly electron-accepting", "D. Less hydrophobic than hydrogen and weakly electron-releasing"]
gb.answer("Q1", response[:1])

In [ ]:
#@title Q2   (run after choosing) { display-mode: "form" }
#@markdown **Q2.**  Why is biological activity entered into a QSAR equation as log(1/C)?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Because concentration terms must be dimensionless before regression", "B. Because the reciprocal converts an EC50 value into a Ki value", "C. Because logarithms remove the need to report standard deviations", "D. Because potency spans orders of magnitude and rises as C falls"]
gb.answer("Q2", response[:1])

In [ ]:
#@title Q3   (run after choosing) { display-mode: "form" }
#@markdown **Q3.**  A series of ethers fits log(1/C) = -0.22(log P)^2 + 1.04 log P + 2.16. What does the negative squared term indicate?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The data were fitted with too few compounds to be reliable", "B. Activity falls steadily as hydrophobicity increases throughout", "C. Activity peaks at an optimum hydrophobicity and then declines", "D. Hydrophobicity makes no measurable contribution to the activity"]
gb.answer("Q3", response[:1])

In [ ]:
#@title Q4   (run after choosing) { display-mode: "form" }
#@markdown **Q4.**  Hammett sigma values are tabulated for meta and para substituents but not ortho. Why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Ortho substituents cannot transmit inductive effects to the acid", "B. Steric interference at ortho positions corrupts the measurement", "C. Ortho positions are not conjugated with the carboxylate group", "D. Ortho-substituted benzoic acids are too insoluble to titrate"]
gb.answer("Q4", response[:1])

In [ ]:
#@title Q5   (run after choosing) { display-mode: "form" }
#@markdown **Q5.**  Taft's Es for hydrogen is 0.00 and for tert-butyl is -2.78. What does the more negative value mean?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The group donates electron density more strongly than hydrogen", "B. The group hinders the reference reaction more than hydrogen does", "C. The group lowers the molar refractivity of the parent compound", "D. The group reduces the compound's aqueous solubility more sharply"]
gb.answer("Q5", response[:1])

## Your results

In [ ]:
#@title Show my performance and my report code { display-mode: "form" }
gb.summary()
print()
_ = gb.report_code()

### What to submit

1. Copy the single `L29|...` line printed above.
2. Paste it into the Canvas assignment **L29 Colab activity**.
3. With your partner, be ready to say one thing the simulation showed you that
   the reading did not.

Your report code records how many questions you answered correctly. It is a
participation and self-check record, not a proctored exam score.